In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

In [5]:
import pandas as pd
import numpy as np

from src.customer_state import generate_customer_state
from src.usage import generate_usage
from src.config import RANDOM_SEED

In [6]:
customers = pd.read_csv(
    "../data/raw/customers.csv",
    parse_dates=["signup_date"],
)

contracts = pd.read_csv(
    "../data/raw/contracts.csv",
    parse_dates=[
        "start_date",
        "end_date",
        "renewal_date",
    ],
)

subscriptions = pd.read_csv(
    "../data/raw/subscriptions.csv",
    parse_dates=[
        "start_date",
        "end_date",
    ],
)

In [10]:
customer_state = generate_customer_state(
    customers=customers,
    start_date="2023-01-02",
    end_date="2026-01-05",
    seed=RANDOM_SEED,
)

In [5]:
customer_state.shape

(3160000, 7)

In [6]:
customer_state.head()

,customer_id,week_start,health_score,baseline_health,health_trend,volatility,behaviour
0,C00001,2023-01-02,77.49,78.05,-0.0515,0.8738,Stable
1,C00001,2023-01-09,78.37,78.05,-0.0515,0.8738,Stable
2,C00001,2023-01-16,79.90,78.05,-0.0515,0.8738,Stable
3,C00001,2023-01-23,79.25,78.05,-0.0515,0.8738,Stable
4,C00001,2023-01-30,78.64,78.05,-0.0515,0.8738,Stable


In [7]:
customer_state["week_start"].min(), \
customer_state["week_start"].max()

(Timestamp('2023-01-02 00:00:00'), Timestamp('2026-01-05 00:00:00'))

In [8]:
customer_state.memory_usage(
    deep=True
).sum() / 1024**2

np.float64(209.7682342529297)

In [9]:
customer_state.dtypes

customer_id                   str
week_start         datetime64[us]
health_score              float64
baseline_health           float64
health_trend              float64
volatility                float64
behaviour                     str
dtype: object

In [11]:
for col in [
    "behaviour",
]:
    if col in customer_state.columns:
        customer_state[col] = (
            customer_state[col]
            .astype("category")
        )

In [12]:
customer_state["customer_id"] = (
    customer_state["customer_id"]
    .astype("string")
)

In [13]:
usage = generate_usage(
    customers=customers,
    subscriptions=subscriptions,
    customer_state=customer_state,
    seed=RANDOM_SEED,
)

In [13]:
usage.shape

(3160000, 12)

In [14]:
usage.info()


<class 'pandas.DataFrame'>
RangeIndex: 3160000 entries, 0 to 3159999
Data columns (total 12 columns):
 #   Column         Dtype         
---  ------         -----         
 0   customer_id    object        
 1   week_start     datetime64[us]
 2   active_users   int64         
 3   sessions       int64         
 4   active_days    int64         
 5   usage_minutes  int64         
 6   features_used  int64         
 7   p001_sessions  int64         
 8   p002_sessions  int64         
 9   p003_sessions  int64         
 10  p004_sessions  int64         
 11  p005_sessions  int64         
dtypes: datetime64[us](1), int64(10), object(1)
memory usage: 289.3+ MB


In [15]:
usage.head()

,customer_id,week_start,active_users,sessions,active_days,usage_minutes,features_used,p001_sessions,p002_sessions,p003_sessions,p004_sessions,p005_sessions
0,C00001,2023-01-02,0,0,0,0,0,0,0,0,0,0
1,C00001,2023-01-09,0,0,0,0,0,0,0,0,0,0
2,C00001,2023-01-16,0,0,0,0,0,0,0,0,0,0
3,C00001,2023-01-23,0,0,0,0,0,0,0,0,0,0
4,C00001,2023-01-30,0,0,0,0,0,0,0,0,0,0


In [16]:
usage[
    [
        "active_users",
        "sessions",
        "active_days",
        "usage_minutes",
        "features_used",
    ]
].describe().round(2)

,active_users,sessions,active_days,usage_minutes,features_used
count,3160000.00,3160000.00,3160000.00,3160000.00,3160000.00
mean,53.91,393.26,1.07,7566.94,0.46
std,368.97,3008.90,2.31,58081.88,1.07
min,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00,0.00
50%,0.00,0.00,0.00,0.00,0.00
75%,0.00,0.00,0.00,0.00,0.00
max,20332.00,181133.00,7.00,3817605.00,5.00


In [17]:
for product_id in [
    "P001",
    "P002",
    "P003",
    "P004",
    "P005",
]:

    usage_column = (
        f"{product_id.lower()}_sessions"
    )

    subscribed_customers = set(
        subscriptions.loc[
            subscriptions["product_id"] == product_id,
            "customer_id",
        ]
    )

    invalid_usage = usage[
        (~usage["customer_id"].isin(
            subscribed_customers
        ))
        & (usage[usage_column] > 0)
    ]

    print(
        product_id,
        "invalid rows:",
        len(invalid_usage),
    )

P001 invalid rows: 0
P002 invalid rows: 0
P003 invalid rows: 0
P004 invalid rows: 0
P005 invalid rows: 0


In [18]:
usage["active_days"].between(
    0,
    7,
).all()

np.True_

In [19]:
(usage["active_users"] >= 0).all()

np.True_

In [20]:
(usage["sessions"] >= 0).all()

np.True_

In [21]:
(usage["usage_minutes"] >= 0).all()

np.True_

In [22]:
weekly_activity = (
    usage
    .groupby("week_start")
    .agg(
        customers=("customer_id", "nunique"),
        active_customers=(
            "active_users",
            lambda x: (x > 0).sum(),
        ),
        avg_active_users=(
            "active_users",
            "mean",
        ),
        avg_sessions=(
            "sessions",
            "mean",
        ),
    )
)

In [23]:
weekly_activity.head()

,customers,active_customers,avg_active_users,avg_sessions
week_start,,,,
2023-01-02,20000,0,0.0,0.0
2023-01-09,20000,0,0.0,0.0
2023-01-16,20000,0,0.0,0.0
2023-01-23,20000,0,0.0,0.0
2023-01-30,20000,0,0.0,0.0


In [24]:
weekly_activity.tail()

,customers,active_customers,avg_active_users,avg_sessions
week_start,,,,
2025-12-08,20000,8679,129.80180,949.01095
2025-12-15,20000,8731,131.08700,959.21200
2025-12-22,20000,8799,133.41375,977.81380
2025-12-29,20000,8848,134.37415,983.66470
2026-01-05,20000,8879,136.11690,994.59260


In [25]:
weekly_activity[
    [
        "customers",
        "active_customers",
    ]
].describe().round(2)


,customers,active_customers
count,158.0,158.00
mean,20000.0,3703.41
std,0.0,3400.73
min,20000.0,0.00
25%,20000.0,0.00
50%,20000.0,3362.00
75%,20000.0,7198.00
max,20000.0,8879.00


In [26]:
health_usage = customer_state[
    [
        "customer_id",
        "week_start",
        "health_score",
        "behaviour",
    ]
].merge(
    usage[
        [
            "customer_id",
            "week_start",
            "active_users",
            "sessions",
            "usage_minutes",
        ]
    ],
    on=[
        "customer_id",
        "week_start",
    ],
    how="inner",
)

In [27]:
health_usage.groupby(
    "behaviour"
)[
    [
        "health_score",
        "active_users",
        "sessions",
        "usage_minutes",
    ]
].mean().round(2)

,health_score,active_users,sessions,usage_minutes
behaviour,,,,
Declining,55.03,25.89,165.87,3203.51
Improving,91.35,90.74,720.78,13878.11
Rapidly Declining,36.07,12.50,63.43,1220.69
Stable,74.82,57.62,415.62,7990.83


In [28]:
usage.memory_usage(
    deep=True
).sum() / 1024**2

np.float64(430.946475982666)

In [29]:
usage.shape

(3160000, 12)

In [30]:
usage.to_parquet(
    "../data/processed/usage.parquet",
    index=False,
)

In [32]:
import os

os.path.getsize(
    "../data/processed/usage.parquet"
) / 1024**2

7.938753128051758

In [35]:
usage.sample(
    10_000,
    random_state=42,
).to_csv(
    "../data/samples/usage_sample.csv",
    index=False,
)

In [3]:
from src.support import generate_support

In [8]:
test_customers = customers.head(100).copy()

In [17]:
test_state = customer_state.head(100).copy()

In [15]:
test_subscriptions= subscriptions.head(100).copy()

In [18]:
test_support = generate_support(
    customers=test_customers,
    customer_state=test_state,
    subscriptions=test_subscriptions,
    seed=42,
)

In [19]:
from src.customers import generate_customers
from src.contracts import generate_contracts
from src.products import generate_products
from src.subscriptions import generate_subscriptions
from src.customer_state import generate_customer_state

In [28]:
test_customers = generate_customers(
    n_customers=1200,
    seed=42,
)

print(test_customers.shape)

(1200, 8)


In [21]:
test_customers.shape

(1200, 8)

In [29]:
test_contracts = generate_contracts(
    customers=test_customers,
    seed=42,
)

print(test_contracts.shape)

(1200, 11)


In [30]:
test_products = generate_products()

print(test_products.shape)

(5, 4)


In [32]:
test_subscriptions = generate_subscriptions(
    customers=test_customers,
    contracts=test_contracts,
    seed=42,
)

print(test_subscriptions.shape)

(2970, 8)


In [33]:
test_state = generate_customer_state(
    customers=test_customers,
    start_date="2026-01-05",
    end_date="2026-06-22",
    seed=42,
)

print(test_state.shape)

(30000, 7)


In [34]:
print("Customers:", test_customers["customer_id"].nunique())
print("Subscriptions:", test_subscriptions["customer_id"].nunique())
print("Customer State:", test_state["customer_id"].nunique())

print("\nCustomer IDs match:",
      set(test_customers["customer_id"])
      == set(test_subscriptions["customer_id"])
      == set(test_state["customer_id"]))

print("\nState period:")
print(test_state["week_start"].min())
print(test_state["week_start"].max())

Customers: 1200
Subscriptions: 1200
Customer State: 1200

Customer IDs match: True

State period:
2026-01-05 00:00:00
2026-06-22 00:00:00


In [35]:
from src.support import generate_support

In [36]:
test_support = generate_support(
    customers=test_customers,
    customer_state=test_state,
    subscriptions=test_subscriptions,
    seed=42,
)

In [37]:
print(test_support.shape)
test_support.head()


(2506, 11)


,ticket_id,customer_id,created_at,resolved_at,issue_category,priority,channel,resolution_hours,reopened,csat_score,escalated
0,TKT00000001,C00002,2026-01-24 16:30:00,2026-01-26 13:00:00,Integration,Medium,Support Portal,44.50,False,4,False
1,TKT00000002,C00002,2026-02-16 16:11:00,2026-02-21 01:03:48,Integration,Low,Email,104.88,False,4,False
2,TKT00000003,C00004,2026-04-16 10:45:00,2026-04-17 19:34:12,Feature Request,High,Chat,32.82,False,4,True
3,TKT00000004,C00005,2026-01-25 13:46:00,2026-01-27 00:28:00,Technical Issue,Medium,Chat,34.70,False,4,False
4,TKT00000005,C00005,2026-02-11 14:18:00,2026-02-17 19:12:00,Performance,Low,Phone,148.90,False,3,False


In [38]:
test_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 2506 entries, 0 to 2505
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ticket_id         2506 non-null   str           
 1   customer_id       2506 non-null   str           
 2   created_at        2506 non-null   datetime64[us]
 3   resolved_at       2506 non-null   datetime64[ns]
 4   issue_category    2506 non-null   str           
 5   priority          2506 non-null   str           
 6   channel           2506 non-null   str           
 7   resolution_hours  2506 non-null   float64       
 8   reopened          2506 non-null   bool          
 9   csat_score        2506 non-null   int64         
 10  escalated         2506 non-null   bool          
dtypes: bool(2), datetime64[ns](1), datetime64[us](1), float64(1), int64(1), str(5)
memory usage: 284.7 KB


In [49]:
test_support["issue_category"].value_counts(normalize=True).round(3)

issue_category
Technical Issue          0.234
Product Bug              0.168
Performance              0.129
Integration              0.117
Billing                  0.104
Feature Request          0.100
Account Management       0.079
Training / Onboarding    0.069
Name: proportion, dtype: float64

In [50]:
test_support["priority"].value_counts(normalize=True).round(3)

priority
Medium      0.482
High        0.241
Low         0.214
Critical    0.063
Name: proportion, dtype: float64

In [51]:
test_support["channel"].value_counts(normalize=True).round(3)

channel
Chat              0.302
Email             0.294
Phone             0.205
Support Portal    0.199
Name: proportion, dtype: float64

In [52]:
test_support[[
    "resolution_hours",
    "csat_score"
]].describe()

,resolution_hours,csat_score
count,2506.000000,2506.000000
mean,40.654665,4.086193
std,36.806738,0.649699
min,1.030000,2.000000
25%,16.580000,4.000000
50%,30.160000,4.000000
75%,52.142500,5.000000
max,328.310000,5.000000


In [53]:
print("Reopen rate:",
      test_support["reopened"].mean().round(3))

print("Escalation rate:",
      test_support["escalated"].mean().round(3))

Reopen rate: 0.087
Escalation rate: 0.108


In [59]:
support_health = test_support.copy()

support_health["week_start"] = (
    support_health["created_at"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

support_health = support_health.merge(
    test_state[
        ["customer_id", "week_start", "health_score", "behaviour"]
    ],
    on=["customer_id", "week_start"],
    how="left",
)

print(
    "Tickets without health match:",
    support_health["health_score"].isna().sum()
)

Tickets without health match: 0


In [60]:
support_health.groupby("priority")["health_score"].mean().round(2)

priority
Critical    71.34
High        71.79
Low         71.18
Medium      71.67
Name: health_score, dtype: float64

In [61]:
support_health[[
    "health_score",
    "csat_score"
]].corr().round(2)

,health_score,csat_score
health_score,1.00,0.22
csat_score,0.22,1.00


In [62]:
support_health.groupby("behaviour").agg(
    tickets=("ticket_id", "count"),
    avg_resolution_hours=("resolution_hours", "mean"),
    avg_csat=("csat_score", "mean"),
    escalation_rate=("escalated", "mean"),
    reopen_rate=("reopened", "mean"),
).round(2)

,tickets,avg_resolution_hours,avg_csat,escalation_rate,reopen_rate
behaviour,,,,,
Declining,555,41.03,4.04,0.10,0.12
Improving,295,40.80,4.09,0.10,0.06
Rapidly Declining,131,43.26,4.01,0.17,0.11
Stable,1525,40.27,4.11,0.11,0.08


In [63]:
support_health.groupby("behaviour").agg(
    customer_weeks=("customer_id", "size"),
    unique_customers=("customer_id", "nunique"),
    tickets=("ticket_id", "count"),
).assign(
    tickets_per_customer_week=lambda x:
        x["tickets"] / x["customer_weeks"]
).round(3)

,customer_weeks,unique_customers,tickets,tickets_per_customer_week
behaviour,,,,
Declining,555,132,555,1.0
Improving,295,70,295,1.0
Rapidly Declining,131,34,131,1.0
Stable,1525,378,1525,1.0


In [64]:
support_health.groupby(
    pd.cut(
        support_health["health_score"],
        bins=[0, 40, 60, 80, 100],
        labels=["Very Low", "Low", "Medium", "High"]
    )
).agg(
    tickets=("ticket_id", "count"),
    avg_csat=("csat_score", "mean"),
    escalation_rate=("escalated", "mean"),
    reopen_rate=("reopened", "mean"),
).round(3)

,tickets,avg_csat,escalation_rate,reopen_rate
health_score,,,,
Very Low,1,3.000,0.000,0.000
Low,330,3.900,0.124,0.118
Medium,1696,4.062,0.110,0.082
High,479,4.303,0.088,0.081


In [65]:
active_state = test_state.copy()

active_state["is_active"] = True

In [66]:
tickets_by_behaviour = (
    support_health
    .groupby("behaviour")
    .size()
    .rename("tickets")
)

weeks_by_behaviour = (
    active_state
    .groupby("behaviour")
    .size()
    .rename("customer_weeks")
)

ticket_rate = pd.concat(
    [tickets_by_behaviour, weeks_by_behaviour],
    axis=1
)

ticket_rate["tickets_per_customer_week"] = (
    ticket_rate["tickets"] / ticket_rate["customer_weeks"]
)

ticket_rate.round(3)

,tickets,customer_weeks,tickets_per_customer_week
behaviour,,,
Declining,555,6250,0.089
Improving,295,4350,0.068
Rapidly Declining,131,1625,0.081
Stable,1525,17775,0.086


In [67]:
state_health = test_state.copy()

state_health["health_band"] = pd.cut(
    state_health["health_score"],
    bins=[0, 40, 60, 80, 100],
    labels=["Very Low", "Low", "Medium", "High"],
    include_lowest=True,
)

health_customer_weeks = (
    state_health
    .groupby("health_band", observed=False)
    .size()
    .rename("customer_weeks")
)

health_tickets = (
    support_health
    .groupby(
        pd.cut(
            support_health["health_score"],
            bins=[0, 40, 60, 80, 100],
            labels=["Very Low", "Low", "Medium", "High"],
            include_lowest=True,
        ),
        observed=False,
    )
    .size()
    .rename("tickets")
)

health_ticket_rate = pd.concat(
    [health_tickets, health_customer_weeks],
    axis=1,
)

health_ticket_rate["tickets_per_customer_week"] = (
    health_ticket_rate["tickets"]
    / health_ticket_rate["customer_weeks"]
)

health_ticket_rate.round(3)

,tickets,customer_weeks,tickets_per_customer_week
Very Low,1,28,0.036
Low,330,2475,0.133
Medium,1696,18555,0.091
High,479,8942,0.054


In [68]:
test_support = generate_support(
    customers=test_customers,
    customer_state=test_state,
    subscriptions=test_subscriptions,
    seed=42,
)

In [69]:
print(test_support.shape)

(2506, 11)


In [70]:
support_health = test_support.copy()

support_health["week_start"] = (
    support_health["created_at"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

support_health = support_health.merge(
    test_state[
        [
            "customer_id",
            "week_start",
            "health_score",
            "behaviour",
        ]
    ],
    on=["customer_id", "week_start"],
    how="left",
)

print(
    "Tickets without health match:",
    support_health["health_score"].isna().sum()
)

Tickets without health match: 0


In [71]:
support_health.groupby(
    "priority"
)["health_score"].mean().round(2)

priority
Critical    71.34
High        71.79
Low         71.18
Medium      71.67
Name: health_score, dtype: float64

In [72]:
support_health.groupby(
    "priority"
).size()

priority
Critical     157
High         604
Low          536
Medium      1209
dtype: int64

In [73]:
import pandas as pd

from src.support import generate_support

In [75]:
customers = pd.read_csv(
    "../data/raw/customers.csv",
    parse_dates=["signup_date"],
)

contracts = pd.read_csv(
    "../data/raw/contracts.csv",
)

products = pd.read_csv(
    "../data/raw/products.csv",
)

subscriptions = pd.read_csv(
    "../data/raw/subscriptions.csv",
)

In [76]:
from src.customer_state import generate_customer_state

customer_state = generate_customer_state(
    customers=customers,
    start_date="2023-01-02",
    end_date="2026-01-05",
    seed=42,
)

In [77]:
print(customer_state.shape)
print(customer_state["week_start"].min())
print(customer_state["week_start"].max())

(3160000, 7)
2023-01-02 00:00:00
2026-01-05 00:00:00


In [78]:
support = generate_support(
    customers=customers,
    customer_state=customer_state,
    subscriptions=subscriptions,
    seed=42,
)

In [79]:
print(support.shape)
support.head()

(114755, 11)


,ticket_id,customer_id,created_at,resolved_at,issue_category,priority,channel,resolution_hours,reopened,csat_score,escalated
0,TKT00000001,C00001,2024-10-12 16:30:00,2024-10-14 13:00:00,Integration,Medium,Support Portal,44.50,False,4,False
1,TKT00000002,C00001,2024-11-04 16:11:00,2024-11-09 01:03:48,Integration,Low,Email,104.88,False,4,False
2,TKT00000003,C00001,2024-12-14 12:53:00,2024-12-17 10:09:12,Performance,Low,Email,69.27,False,3,False
3,TKT00000004,C00001,2025-02-12 17:03:00,2025-02-13 22:45:00,Product Bug,Medium,Phone,29.70,False,4,False
4,TKT00000005,C00001,2025-04-17 10:45:00,2025-04-17 19:30:00,Billing,High,Chat,8.75,False,4,True


In [80]:
support.to_parquet(
    "../data/processed/support.parquet",
    index=False,
)

In [81]:
support.sample(
    min(5000, len(support)),
    random_state=42,
).to_csv(
    "../data/samples/support_sample.csv",
    index=False,
)

In [82]:
print("Support rows:", len(support))
print("Unique customers:", support["customer_id"].nunique())
print("Date range:", support["created_at"].min(), "→", support["created_at"].max())
print("Saved to: data/processed/support.parquet")

Support rows: 114755
Unique customers: 12493
Date range: 2024-01-04 17:00:00 → 2026-01-11 18:57:00
Saved to: data/processed/support.parquet


In [84]:
usage = pd.read_parquet(
    "../data/processed/usage.parquet"
)

print("Usage shape:", usage.shape)
print("Usage start:", usage["week_start"].min())
print("Usage end:", usage["week_start"].max())
print("Unique customers:", usage["customer_id"].nunique())

Usage shape: (3160000, 12)
Usage start: 2023-01-02 00:00:00
Usage end: 2026-01-05 00:00:00
Unique customers: 20000


In [85]:
from src.customer_state import generate_customer_state

customer_state = generate_customer_state(
    customers=customers,
    start_date="2023-01-02",
    end_date="2026-01-05",
    seed=42,
)

In [86]:
print("Customer state shape:", customer_state.shape)
print("Unique customers:", customer_state["customer_id"].nunique())
print("Start:", customer_state["week_start"].min())
print("End:", customer_state["week_start"].max())

Customer state shape: (3160000, 7)
Unique customers: 20000
Start: 2023-01-02 00:00:00
End: 2026-01-05 00:00:00


In [87]:
customer_state["behaviour"].value_counts(normalize=True).round(3)

behaviour
Stable               0.598
Declining            0.202
Improving            0.150
Rapidly Declining    0.050
Name: proportion, dtype: float64

In [89]:
customer_state.to_parquet(
    "../data/processed/customer_state.parquet",
    index=False,
)

In [91]:
import os

print(
    "Saved:",
    os.path.exists("../data/processed/customer_state.parquet")
)

Saved: True


In [92]:
customer_state.head()

,customer_id,week_start,health_score,baseline_health,health_trend,volatility,behaviour
0,C00001,2023-01-02,77.49,78.05,-0.0515,0.8738,Stable
1,C00001,2023-01-09,78.37,78.05,-0.0515,0.8738,Stable
2,C00001,2023-01-16,79.90,78.05,-0.0515,0.8738,Stable
3,C00001,2023-01-23,79.25,78.05,-0.0515,0.8738,Stable
4,C00001,2023-01-30,78.64,78.05,-0.0515,0.8738,Stable


In [93]:
customer_state.info()

<class 'pandas.DataFrame'>
RangeIndex: 3160000 entries, 0 to 3159999
Data columns (total 7 columns):
 #   Column           Dtype         
---  ------           -----         
 0   customer_id      str           
 1   week_start       datetime64[us]
 2   health_score     float64       
 3   baseline_health  float64       
 4   health_trend     float64       
 5   volatility       float64       
 6   behaviour        str           
dtypes: datetime64[us](1), float64(4), str(2)
memory usage: 209.8 MB


In [94]:
from src.support import generate_support

support = generate_support(
    customers=customers,
    customer_state=customer_state,
    subscriptions=subscriptions,
    seed=42,
)

In [95]:
print("Support shape:", support.shape)
print("Unique customers:", support["customer_id"].nunique())
print("Date range:", support["created_at"].min(), "→", support["created_at"].max())

Support shape: (114755, 11)
Unique customers: 12493
Date range: 2024-01-04 17:00:00 → 2026-01-11 18:57:00


In [96]:
support.info()

<class 'pandas.DataFrame'>
RangeIndex: 114755 entries, 0 to 114754
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   ticket_id         114755 non-null  str           
 1   customer_id       114755 non-null  str           
 2   created_at        114755 non-null  datetime64[us]
 3   resolved_at       114755 non-null  datetime64[ns]
 4   issue_category    114755 non-null  str           
 5   priority          114755 non-null  str           
 6   channel           114755 non-null  str           
 7   resolution_hours  114755 non-null  float64       
 8   reopened          114755 non-null  bool          
 9   csat_score        114755 non-null  int64         
 10  escalated         114755 non-null  bool          
dtypes: bool(2), datetime64[ns](1), datetime64[us](1), float64(1), int64(1), str(5)
memory usage: 12.7 MB


In [97]:
print("Support shape:", support.shape)
print("Unique customers:", support["customer_id"].nunique())
print("Duplicate ticket IDs:", support["ticket_id"].duplicated().sum())
print("Missing values:", support.isna().sum().sum())

Support shape: (114755, 11)
Unique customers: 12493
Duplicate ticket IDs: 0
Missing values: 0


In [98]:
print("Issue categories:")
print(
    support["issue_category"]
    .value_counts(normalize=True)
    .round(3)
)

print("\nPriority:")
print(
    support["priority"]
    .value_counts(normalize=True)
    .round(3)
)

print("\nChannels:")
print(
    support["channel"]
    .value_counts(normalize=True)
    .round(3)
)

Issue categories:
issue_category
Technical Issue          0.252
Product Bug              0.171
Performance              0.136
Integration              0.126
Billing                  0.097
Feature Request          0.083
Account Management       0.077
Training / Onboarding    0.058
Name: proportion, dtype: float64

Priority:
priority
Medium      0.470
High        0.245
Low         0.219
Critical    0.066
Name: proportion, dtype: float64

Channels:
channel
Chat              0.302
Email             0.299
Support Portal    0.200
Phone             0.199
Name: proportion, dtype: float64


In [99]:
print(
    support[
        ["resolution_hours", "csat_score"]
    ].describe().round(2)
)

print(
    "\nReopen rate:",
    round(support["reopened"].mean(), 3)
)

print(
    "Escalation rate:",
    round(support["escalated"].mean(), 3)
)

       resolution_hours  csat_score
count         114755.00   114755.00
mean              39.90        3.96
std               36.31        0.69
min                0.50        1.00
25%               16.83        4.00
50%               29.62        4.00
75%               50.72        4.00
max              589.00        5.00

Reopen rate: 0.083
Escalation rate: 0.117


In [100]:
support_health = support.copy()

support_health["week_start"] = (
    support_health["created_at"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

support_health = support_health.merge(
    customer_state[
        [
            "customer_id",
            "week_start",
            "health_score",
            "behaviour",
        ]
    ],
    on=["customer_id", "week_start"],
    how="left",
)

print(
    "Tickets without health match:",
    support_health["health_score"].isna().sum()
)

Tickets without health match: 0


In [101]:
print(
    support_health[
        ["health_score", "csat_score"]
    ].corr().round(2)
)

              health_score  csat_score
health_score          1.00        0.36
csat_score            0.36        1.00


In [102]:
support_health.groupby(
    pd.cut(
        support_health["health_score"],
        bins=[0, 40, 60, 80, 100],
        labels=["Very Low", "Low", "Medium", "High"],
        include_lowest=True,
    ),
    observed=False,
).agg(
    tickets=("ticket_id", "count"),
    avg_csat=("csat_score", "mean"),
    escalation_rate=("escalated", "mean"),
    reopen_rate=("reopened", "mean"),
).round(3)

,tickets,avg_csat,escalation_rate,reopen_rate
health_score,,,,
Very Low,25579,3.589,0.139,0.085
Low,27584,3.890,0.120,0.083
Medium,37060,4.066,0.111,0.084
High,24532,4.279,0.098,0.081


In [103]:
support_health.groupby("behaviour").agg(
    tickets=("ticket_id", "count"),
    avg_resolution_hours=("resolution_hours", "mean"),
    avg_csat=("csat_score", "mean"),
    escalation_rate=("escalated", "mean"),
    reopen_rate=("reopened", "mean"),
).round(2)

,tickets,avg_resolution_hours,avg_csat,escalation_rate,reopen_rate
behaviour,,,,,
Declining,33778,39.00,3.79,0.13,0.08
Improving,9440,41.99,4.33,0.10,0.08
Rapidly Declining,10081,39.02,3.53,0.14,0.09
Stable,61456,40.22,4.08,0.11,0.08


In [104]:
active_customers = (
    customer_state[
        ["customer_id", "week_start"]
    ]
    .merge(
        subscriptions[
            ["customer_id", "start_date", "end_date"]
        ]
        .assign(
            start_date=lambda df: pd.to_datetime(df["start_date"]),
            end_date=lambda df: pd.to_datetime(df["end_date"]),
        ),
        on="customer_id",
        how="left",
    )
)

active_customers["is_active"] = (
    (active_customers["week_start"] >= active_customers["start_date"])
    & (active_customers["week_start"] <= active_customers["end_date"])
)

active_customer_weeks = (
    active_customers
    .groupby("customer_id")["is_active"]
    .sum()
)

print("Customers:", len(active_customer_weeks))
print("Customers with at least 1 active week:",
      (active_customer_weeks > 0).sum())
print("Customers with >= 26 active weeks:",
      (active_customer_weeks >= 26).sum())
print("Customers with >= 52 active weeks:",
      (active_customer_weeks >= 52).sum())

print("\nActive weeks per customer:")
print(active_customer_weeks.describe())

Customers: 20000
Customers with at least 1 active week: 13442
Customers with >= 26 active weeks: 11795
Customers with >= 52 active weeks: 10068

Active weeks per customer:
count    20000.000000
mean        72.452700
std         83.073585
min          0.000000
25%          0.000000
50%         52.000000
75%        106.000000
max        520.000000
Name: is_active, dtype: float64


In [105]:
print("Customers with zero active weeks:")
print(
    active_customer_weeks[
        active_customer_weeks == 0
    ].describe()
)

Customers with zero active weeks:
count    6558.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: is_active, dtype: float64


In [106]:
contract_dates_check = (
    subscriptions[
        ["customer_id", "start_date", "end_date"]
    ]
    .assign(
        start_date=lambda df: pd.to_datetime(df["start_date"]),
        end_date=lambda df: pd.to_datetime(df["end_date"]),
    )
    .groupby("customer_id")
    .agg(
        contract_start=("start_date", "min"),
        contract_end=("end_date", "max"),
    )
)

zero_active = active_customer_weeks[
    active_customer_weeks == 0
].index

contract_dates_check.loc[
    contract_dates_check.index.isin(zero_active)
].head(20)

,contract_start,contract_end
customer_id,,
C00005,2026-03-29,2027-03-29
C00011,2026-04-13,2027-04-13
C00014,2026-05-26,2027-05-26
C00015,2026-03-17,2029-03-16
C00021,2026-10-29,2029-10-28
C00022,2026-11-09,2027-11-09
C00023,2026-07-21,2027-07-21
C00029,2026-11-16,2027-11-16
C00031,2026-02-08,2027-02-08


In [107]:
print(
    contract_dates_check.loc[
        contract_dates_check.index.isin(zero_active)
    ].describe()
)

                   contract_start                contract_end
count                        6558                        6558
mean   2026-07-04 17:25:24.977127  2027-11-27 17:09:23.220494
min           2026-01-06 00:00:00         2027-01-06 00:00:00
25%           2026-04-04 00:00:00         2027-05-15 00:00:00
50%           2026-07-04 00:00:00         2027-09-27 00:00:00
75%           2026-10-05 00:00:00         2028-04-11 18:00:00
max           2026-12-31 00:00:00         2029-12-30 00:00:00


In [109]:
support.to_parquet(
    "../data/processed/support.parquet",
    index=False,
)

In [111]:
support.sample(
    min(5000, len(support)),
    random_state=42,
).to_csv(
    "../data/samples/support_sample.csv",
    index=False,
)

In [113]:
import os

print(
    "Support parquet saved:",
    os.path.exists("../data/processed/support.parquet")
)

print(
    "Support sample saved:",
    os.path.exists("../data/samples/support_sample.csv")
)

Support parquet saved: True
Support sample saved: True


In [114]:
contracts.columns.tolist()

['contract_id',
 'customer_id',
 'segment',
 'start_date',
 'end_date',
 'renewal_date',
 'annual_contract_value',
 'total_contract_value',
 'contract_type',
 'auto_renew',
 'contract_status']

In [115]:
contracts.head()

,contract_id,customer_id,segment,start_date,end_date,renewal_date,annual_contract_value,total_contract_value,contract_type,auto_renew,contract_status
0,CT000001,C00001,Mid-Market,2024-09-19,2027-09-19,2027-09-19,1000000.00,3000000.00,Three-year,True,Active
1,CT000002,C00002,SMB,2025-05-22,2027-05-22,2027-05-22,386859.14,773718.28,Two-year,True,Active
2,CT000003,C00003,Enterprise,2024-09-24,2026-09-24,2026-09-24,7500000.00,15000000.00,Two-year,True,Active
3,CT000004,C00004,Mid-Market,2024-02-15,2027-02-14,2027-02-14,10000000.00,30000000.00,Three-year,False,Active
4,CT000005,C00005,SMB,2026-03-29,2027-03-29,2027-03-29,574056.25,574056.25,Annual,True,Active


In [116]:
contracts[
    ["contract_type", "auto_renew"]
].value_counts(normalize=True)

contract_type  auto_renew
Annual         True          0.50355
               False         0.17850
Two-year       True          0.17225
Three-year     True          0.06825
Two-year       False         0.05775
Three-year     False         0.01970
Name: proportion, dtype: float64

In [117]:
from src.churn import generate_churn

In [118]:
test_churn = generate_churn(
    customers= test_customers,
    contracts= test_contracts,
    customer_state= test_state,
    usage= usage,
    support= test_support,
    seed= 42
)

In [119]:
print("Churn shape:", test_churn.shape)
print("Unique customers:", test_churn["customer_id"].nunique())

test_churn.head()

Churn shape: (122, 7)
Unique customers: 122


,customer_id,contract_id,churn_date,churn_reason,churn_probability,health_score_at_churn,behaviour_at_churn
0,C00608,CT000608,2025-11-27,Poor Customer Experience,0.3661,79.76,Declining
1,C00540,CT000540,2025-12-07,Low Product Engagement,0.2728,67.73,Stable
2,C00161,CT000161,2025-12-19,Poor Customer Experience,0.1153,65.03,Stable
3,C00752,CT000752,2026-01-21,Low Product Engagement,0.0629,73.46,Stable
4,C00136,CT000136,2026-02-06,Product / Technical Issues,0.2095,64.15,Declining


In [120]:
test_churn.info()

<class 'pandas.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   customer_id            122 non-null    str           
 1   contract_id            122 non-null    str           
 2   churn_date             122 non-null    datetime64[us]
 3   churn_reason           122 non-null    str           
 4   churn_probability      122 non-null    float64       
 5   health_score_at_churn  122 non-null    float64       
 6   behaviour_at_churn     122 non-null    str           
dtypes: datetime64[us](1), float64(2), str(4)
memory usage: 12.5 KB


In [121]:
print(
    "Churn rate:",
    round(
        test_churn["customer_id"].nunique()
        / test_customers["customer_id"].nunique(),
        3,
    )
)

Churn rate: 0.102


In [122]:
test_churn.merge(
    test_customers[
        ["customer_id", "segment"]
    ],
    on="customer_id",
    how="left",
).groupby("segment").agg(
    churned_customers=("customer_id", "nunique"),
).assign(
    churn_rate=lambda x:
        x["churned_customers"]
        / test_customers["segment"].value_counts()
)

,churned_customers,churn_rate
segment,,
Enterprise,6,0.032967
Mid-Market,26,0.071233
SMB,90,0.137825


In [123]:
test_churn["behaviour_at_churn"].value_counts(
    normalize=True
).round(3)

behaviour_at_churn
Stable               0.508
Declining            0.311
Rapidly Declining    0.148
Improving            0.033
Name: proportion, dtype: float64

In [124]:
print(
    "Earliest churn:",
    test_churn["churn_date"].min()
)

print(
    "Latest churn:",
    test_churn["churn_date"].max()
)

Earliest churn: 2025-11-27 00:00:00
Latest churn: 2029-08-08 00:00:00


In [125]:
print(
    "Churns inside observation window:",
    (
        test_churn["churn_date"]
        <= pd.Timestamp("2026-01-05")
    ).sum()
)

Churns inside observation window: 3


In [126]:
test_customers.shape

(1200, 8)

In [127]:
test_support.shape

(2506, 11)

In [128]:
usage.shape

(3160000, 12)

In [140]:
# ============================================
# CREATE CONSISTENT TEST DATASETS
# ============================================

# Use the same 1,200 customers used for previous testing
test_customer_ids = test_customers["customer_id"].unique()

# -----------------------------
# 1. Test Customers
# -----------------------------
test_customers = customers[
    customers["customer_id"].isin(test_customer_ids)
].copy()

# -----------------------------
# 2. Test Contracts
# -----------------------------
test_contracts = contracts[
    contracts["customer_id"].isin(test_customer_ids)
].copy()

# -----------------------------
# 3. Test Customer State
# -----------------------------
test_state = customer_state[
    customer_state["customer_id"].isin(test_customer_ids)
].copy()

# -----------------------------
# 4. Test Usage
# -----------------------------
test_usage = usage[
    usage["customer_id"].isin(test_customer_ids)
].copy()

# -----------------------------
# 5. Test Support
# -----------------------------
test_support = support[
    support["customer_id"].isin(test_customer_ids)
].copy()


# ============================================
# VALIDATION
# ============================================

print("TEST DATASET SIZES")
print("-" * 40)

print("Customers      :", test_customers.shape)
print("Contracts      :", test_contracts.shape)
print("Customer State :", test_state.shape)
print("Usage          :", test_usage.shape)
print("Support        :", test_support.shape)

print("\nUNIQUE CUSTOMERS")
print("-" * 40)

print("Customers      :", test_customers["customer_id"].nunique())
print("Contracts      :", test_contracts["customer_id"].nunique())
print("Customer State :", test_state["customer_id"].nunique())
print("Usage          :", test_usage["customer_id"].nunique())
print("Support        :", test_support["customer_id"].nunique())

TEST DATASET SIZES
----------------------------------------
Customers      : (1200, 8)
Contracts      : (1200, 11)
Customer State : (189600, 7)
Usage          : (189600, 12)
Support        : (7329, 11)

UNIQUE CUSTOMERS
----------------------------------------
Customers      : 1200
Contracts      : 1200
Customer State : 1200
Usage          : 1200
Support        : 761


In [141]:
test_churn = generate_churn(
    customers=test_customers,
    contracts=test_contracts,
    customer_state=test_state,
    usage=test_usage,
    support=test_support,
    seed=42,
)

In [142]:
observation_end = pd.Timestamp("2026-01-05")

observed_test_churn = test_churn[
    test_churn["churn_date"] <= observation_end
].copy()

print("Total churn events:", len(test_churn))
print("Observed churn events:", len(observed_test_churn))

print(
    "Observed churn rate:",
    round(
        len(observed_test_churn) / test_customers["customer_id"].nunique(),
        3
    )
)

print("\nBehaviour at observed churn:")
print(
    observed_test_churn["behaviour_at_churn"]
    .value_counts(normalize=True)
)

Total churn events: 226
Observed churn events: 66
Observed churn rate: 0.055

Behaviour at observed churn:
behaviour_at_churn
Declining            0.469697
Stable               0.257576
Rapidly Declining    0.212121
Improving            0.060606
Name: proportion, dtype: float64


In [133]:
test_churn["behaviour_at_churn"].value_counts(
    normalize=True
).round(3)

behaviour_at_churn
Declining            0.500
Stable               0.283
Rapidly Declining    0.177
Improving            0.040
Name: proportion, dtype: float64

In [134]:
observation_end = pd.Timestamp("2026-01-05")

print("Total churn events:", len(test_churn))

print(
    "Churn events within observation:",
    (test_churn["churn_date"] <= observation_end).sum()
)

print(
    "Churn events after observation:",
    (test_churn["churn_date"] > observation_end).sum()
)

print("\nChurn rate within observation:")
print(
    (test_churn["churn_date"] <= observation_end).sum()
    / test_customers["customer_id"].nunique()
)

Total churn events: 226
Churn events within observation: 66
Churn events after observation: 160

Churn rate within observation:
0.055


In [135]:
test_churn["churn_date"] = pd.to_datetime(test_churn["churn_date"])

print(
    test_churn["churn_date"]
    .dt.year
    .value_counts()
    .sort_index()
)

churn_date
2024     3
2025    61
2026    84
2027    57
2028    18
2029     3
Name: count, dtype: int64


In [136]:
# Customers with at least one active week in the observation period
test_active_customers = test_state.loc[
    test_state["week_start"] <= pd.Timestamp("2026-01-05"),
    "customer_id"
].nunique()

print("Test customers:", test_customers["customer_id"].nunique())
print("Active during observation:", test_active_customers)

# Observed churn among active customers
observed_churn = test_churn.loc[
    test_churn["churn_date"] <= pd.Timestamp("2026-01-05"),
    "customer_id"
].nunique()

print("Observed churn:", observed_churn)

print(
    "Churn rate among active customers:",
    round(observed_churn / test_active_customers, 3)
)

Test customers: 1200
Active during observation: 1200
Observed churn: 66
Churn rate among active customers: 0.055


In [137]:
print(
    test_churn.loc[
        test_churn["churn_date"] <= pd.Timestamp("2026-01-05"),
        "behaviour_at_churn"
    ].value_counts(normalize=True)
)

behaviour_at_churn
Declining            0.469697
Stable               0.257576
Rapidly Declining    0.212121
Improving            0.060606
Name: proportion, dtype: float64


In [151]:
import importlib
import src.churn

importlib.reload(src.churn)

generate_churn = src.churn.generate_churn

In [152]:
print(src.churn.BASE_CHURN_PROBABILITY)

{'SMB': 0.22, 'Mid-Market': 0.15, 'Enterprise': 0.09}


In [153]:
print(src.churn.BASE_CHURN_PROBABILITY)
print(src.churn.BEHAVIOUR_MULTIPLIER)

{'SMB': 0.22, 'Mid-Market': 0.15, 'Enterprise': 0.09}
{'Stable': 0.6, 'Improving': 0.45, 'Declining': 1.7, 'Rapidly Declining': 3.0}


In [154]:
test_churn = generate_churn(
    customers=test_customers,
    contracts=test_contracts,
    customer_state=test_state,
    usage=test_usage,
    support=test_support,
    seed=42,
)

In [155]:
observation_end = pd.Timestamp("2026-01-05")

observed_test_churn = test_churn[
    test_churn["churn_date"] <= observation_end
].copy()

print("Total churn events:", len(test_churn))
print("Observed churn events:", len(observed_test_churn))

print(
    "Observed churn rate:",
    round(
        len(observed_test_churn) / test_customers["customer_id"].nunique(),
        3
    )
)

print("\nBehaviour at observed churn:")
print(
    observed_test_churn["behaviour_at_churn"]
    .value_counts(normalize=True)
)

Total churn events: 167
Observed churn events: 167
Observed churn rate: 0.139

Behaviour at observed churn:
behaviour_at_churn
Declining            0.491018
Stable               0.299401
Rapidly Declining    0.179641
Improving            0.029940
Name: proportion, dtype: float64


In [156]:
print("Earliest:", test_churn["churn_date"].min())
print("Latest:", test_churn["churn_date"].max())
print(
    test_churn["churn_date"]
    .dt.year
    .value_counts()
    .sort_index()
)

Earliest: 2024-02-27 00:00:00
Latest: 2025-12-25 00:00:00
churn_date
2024     57
2025    110
Name: count, dtype: int64


In [157]:
print(
    test_churn
    .merge(
        test_customers[["customer_id", "segment"]],
        on="customer_id",
        how="left"
    )
    ["segment"]
    .value_counts(normalize=True)
)

segment
SMB           0.622754
Mid-Market    0.275449
Enterprise    0.101796
Name: proportion, dtype: float64


In [158]:
segment_summary = (
    test_customers[["customer_id", "segment"]]
    .merge(
        test_churn[["customer_id"]],
        on="customer_id",
        how="left",
        indicator=True
    )
    .assign(
        churned=lambda x: x["_merge"].eq("both")
    )
    .groupby("segment")
    .agg(
        customers=("customer_id", "count"),
        churned=("churned", "sum"),
        churn_rate=("churned", "mean")
    )
)

print(segment_summary)

            customers  churned  churn_rate
segment                                   
Enterprise        182       17    0.093407
Mid-Market        365       46    0.126027
SMB               653      104    0.159265


In [159]:
segment_summary = (
    test_customers[["customer_id", "segment"]]
    .merge(
        test_churn[["customer_id"]],
        on="customer_id",
        how="left",
        indicator=True
    )
    .assign(
        churned=lambda x: x["_merge"].eq("both")
    )
    .groupby("segment")
    .agg(
        customers=("customer_id", "count"),
        churned=("churned", "sum"),
        churn_rate=("churned", "mean")
    )
)

print(segment_summary)

            customers  churned  churn_rate
segment                                   
Enterprise        182       17    0.093407
Mid-Market        365       46    0.126027
SMB               653      104    0.159265


In [160]:
print(
    test_churn["health_score_at_churn"].describe()
)

count    167.000000
mean      46.803832
std       25.430177
min        0.000000
25%       28.445000
50%       46.830000
75%       65.990000
max      100.000000
Name: health_score_at_churn, dtype: float64


In [161]:
print(
    test_churn.groupby("behaviour_at_churn")[
        "health_score_at_churn"
    ].mean()
)

behaviour_at_churn
Declining            39.800244
Improving            94.336000
Rapidly Declining    18.138333
Stable               70.735800
Name: health_score_at_churn, dtype: float64


In [162]:
print(
    test_churn["churn_reason"]
    .value_counts(normalize=True)
)

churn_reason
Low Product Engagement           0.257485
Poor Customer Experience         0.179641
Product / Technical Issues       0.173653
Business / Budget Constraints    0.143713
Contract / Renewal Decision      0.131737
Lack of Required Features        0.113772
Name: proportion, dtype: float64


In [163]:
import importlib
import src.labels

importlib.reload(src.labels)

generate_churn_labels = src.labels.generate_churn_labels

In [164]:
test_labels = generate_churn_labels(
    customers=test_customers,
    churn=test_churn,
    customer_state=test_state,
    observation_end="2026-01-05",
    horizon_days=90,
    seed=42,
)

In [165]:
print("Shape:", test_labels.shape)

print("\nColumns:")
print(test_labels.columns.tolist())

print("\nUnique customers:")
print(test_labels["customer_id"].nunique())

print("\nSnapshot range:")
print(test_labels["snapshot_date"].min())
print(test_labels["snapshot_date"].max())

print("\nLabel distribution:")
print(test_labels["churn_90d"].value_counts())

print("\nLabel proportion:")
print(test_labels["churn_90d"].value_counts(normalize=True))

Shape: (174000, 3)

Columns:
['customer_id', 'snapshot_date', 'churn_90d']

Unique customers:
1200

Snapshot range:
2023-01-02 00:00:00
2025-10-06 00:00:00

Label distribution:
churn_90d
0    171977
1      2023
Name: count, dtype: int64

Label proportion:
churn_90d
0    0.988374
1    0.011626
Name: proportion, dtype: float64


In [166]:
# ---------------------------------------------------------
# BASIC VALIDATION
# ---------------------------------------------------------

assert test_labels["customer_id"].nunique() == 1200

assert test_labels["churn_90d"].isin([0, 1]).all()

assert test_labels["snapshot_date"].max() <= (
    pd.Timestamp("2026-01-05") - pd.Timedelta(days=90)
)

assert test_labels["snapshot_date"].isna().sum() == 0

print("Basic validation: PASSED")

Basic validation: PASSED


In [167]:
churned_ids = test_churn["customer_id"].head(5).tolist()

print(
    test_labels[
        test_labels["customer_id"].isin(churned_ids)
    ].head(50)
)

      customer_id snapshot_date  churn_90d
23780      C00165    2023-01-02          0
23781      C00165    2023-01-09          0
23782      C00165    2023-01-16          0
23783      C00165    2023-01-23          0
23784      C00165    2023-01-30          0
23785      C00165    2023-02-06          0
23786      C00165    2023-02-13          0
23787      C00165    2023-02-20          0
23788      C00165    2023-02-27          0
23789      C00165    2023-03-06          0
23790      C00165    2023-03-13          0
23791      C00165    2023-03-20          0
23792      C00165    2023-03-27          0
23793      C00165    2023-04-03          0
23794      C00165    2023-04-10          0
23795      C00165    2023-04-17          0
23796      C00165    2023-04-24          0
23797      C00165    2023-05-01          0
23798      C00165    2023-05-08          0
23799      C00165    2023-05-15          0
23800      C00165    2023-05-22          0
23801      C00165    2023-05-29          0
23802      

In [168]:
customer_id = churned_ids[0]

print(
    test_labels[
        test_labels["customer_id"] == customer_id
    ].to_string(index=False)
)

customer_id snapshot_date  churn_90d
     C00455    2023-01-02          0
     C00455    2023-01-09          0
     C00455    2023-01-16          0
     C00455    2023-01-23          0
     C00455    2023-01-30          0
     C00455    2023-02-06          0
     C00455    2023-02-13          0
     C00455    2023-02-20          0
     C00455    2023-02-27          0
     C00455    2023-03-06          0
     C00455    2023-03-13          0
     C00455    2023-03-20          0
     C00455    2023-03-27          0
     C00455    2023-04-03          0
     C00455    2023-04-10          0
     C00455    2023-04-17          0
     C00455    2023-04-24          0
     C00455    2023-05-01          0
     C00455    2023-05-08          0
     C00455    2023-05-15          0
     C00455    2023-05-22          0
     C00455    2023-05-29          0
     C00455    2023-06-05          0
     C00455    2023-06-12          0
     C00455    2023-06-19          0
     C00455    2023-06-26          0
 

In [169]:
import importlib
import src.labels

importlib.reload(src.labels)

generate_churn_labels = src.labels.generate_churn_labels

In [170]:
import importlib
import src.labels

importlib.reload(src.labels)

generate_churn_labels = src.labels.generate_churn_labels

In [171]:
test_labels = generate_churn_labels(
    customers=test_customers,
    churn=test_churn,
    customer_state=test_state,
    observation_end="2026-01-05",
    horizon_days=90,
    seed=42,
)


In [172]:
print("Shape:", test_labels.shape)

print("\nColumns:")
print(test_labels.columns.tolist())

print("\nUnique customers:")
print(test_labels["customer_id"].nunique())

print("\nSnapshot range:")
print(test_labels["snapshot_date"].min())
print(test_labels["snapshot_date"].max())

print("\nLabel distribution:")
print(test_labels["churn_90d"].value_counts())

print("\nLabel proportion:")
print(test_labels["churn_90d"].value_counts(normalize=True))

Shape: (169022, 3)

Columns:
['customer_id', 'snapshot_date', 'churn_90d']

Unique customers:
1200

Snapshot range:
2023-01-02 00:00:00
2025-10-06 00:00:00

Label distribution:
churn_90d
0    166999
1      2023
Name: count, dtype: int64

Label proportion:
churn_90d
0    0.988031
1    0.011969
Name: proportion, dtype: float64


In [173]:
validation = test_labels.merge(
    test_churn[["customer_id", "churn_date"]],
    on="customer_id",
    how="left",
)

post_churn_snapshots = validation[
    validation["churn_date"].notna()
    & (
        validation["snapshot_date"]
        >= validation["churn_date"]
    )
]

print(
    "Post-churn snapshots:",
    len(post_churn_snapshots)
)

Post-churn snapshots: 0


In [174]:
positive_labels = test_labels[
    test_labels["churn_90d"] == 1
]

print("Positive rows:", len(positive_labels))

print(
    "Customers with positive labels:",
    positive_labels["customer_id"].nunique()
)

Positive rows: 2023
Customers with positive labels: 167


In [175]:
customer_id = "C00455"

print(
    test_labels[
        test_labels["customer_id"] == customer_id
    ].to_string(index=False)
)

customer_id snapshot_date  churn_90d
     C00455    2023-01-02          0
     C00455    2023-01-09          0
     C00455    2023-01-16          0
     C00455    2023-01-23          0
     C00455    2023-01-30          0
     C00455    2023-02-06          0
     C00455    2023-02-13          0
     C00455    2023-02-20          0
     C00455    2023-02-27          0
     C00455    2023-03-06          0
     C00455    2023-03-13          0
     C00455    2023-03-20          0
     C00455    2023-03-27          0
     C00455    2023-04-03          0
     C00455    2023-04-10          0
     C00455    2023-04-17          0
     C00455    2023-04-24          0
     C00455    2023-05-01          0
     C00455    2023-05-08          0
     C00455    2023-05-15          0
     C00455    2023-05-22          0
     C00455    2023-05-29          0
     C00455    2023-06-05          0
     C00455    2023-06-12          0
     C00455    2023-06-19          0
     C00455    2023-06-26          0
 

In [176]:
positive_customer_check = (
    test_labels[
        test_labels["churn_90d"] == 1
    ]
    ["customer_id"]
    .nunique()
)

churned_customer_count = test_churn["customer_id"].nunique()

print("Churned customers:", churned_customer_count)
print("Customers with positive labels:", positive_customer_check)

Churned customers: 167
Customers with positive labels: 167


In [177]:
positive_window = (
    test_labels[
        test_labels["churn_90d"] == 1
    ]
    .groupby("customer_id")
    .size()
)

print(positive_window.describe())

count    167.000000
mean      12.113772
std        2.218268
min        2.000000
25%       12.000000
50%       13.000000
75%       13.000000
max       13.000000
dtype: float64


In [178]:
full_labels = generate_churn_labels(
    customers=customers,
    churn=churn,
    customer_state=customer_state,
    observation_end="2026-01-05",
    horizon_days=90,
    seed=42,
)

NameError: name 'churn' is not defined

In [180]:
print(customers.shape)
# (20000, 8)

print(contracts.shape)
# (20000, 11)

print(customer_state.shape)
# (3160000, 7)

print(usage.shape)
# (3160000, 12)

support.shape
# (114755, 11)

(20000, 8)
(20000, 11)
(3160000, 7)
(3160000, 12)


(114755, 11)

In [181]:
import importlib
import src.churn

importlib.reload(src.churn)

generate_churn = src.churn.generate_churn

In [182]:
print(src.churn.BASE_CHURN_PROBABILITY)
print(src.churn.BEHAVIOUR_MULTIPLIER)

{'SMB': 0.22, 'Mid-Market': 0.15, 'Enterprise': 0.09}
{'Stable': 0.6, 'Improving': 0.45, 'Declining': 1.7, 'Rapidly Declining': 3.0}


In [183]:
test_churn = generate_churn(
    customers=test_customers,
    contracts=test_contracts,
    customer_state=test_state,
    usage=test_usage,
    support=test_support,
    seed=42,
)


In [184]:
print("Shape:", test_churn.shape)
print("Unique churned customers:", test_churn["customer_id"].nunique())

print(
    "Churn rate:",
    round(
        test_churn["customer_id"].nunique()
        / test_customers["customer_id"].nunique(),
        3
    )
)

print("\nSegment:")
print(
    test_churn
    .merge(
        test_customers[["customer_id", "segment"]],
        on="customer_id",
        how="left"
    )
    ["segment"]
    .value_counts(normalize=True)
)

print("\nBehaviour:")
print(
    test_churn["behaviour_at_churn"]
    .value_counts(normalize=True)
)

print("\nReasons:")
print(
    test_churn["churn_reason"]
    .value_counts(normalize=True)
)

print("\nDates:")
print(test_churn["churn_date"].min())
print(test_churn["churn_date"].max())

Shape: (167, 7)
Unique churned customers: 167
Churn rate: 0.139

Segment:
segment
SMB           0.622754
Mid-Market    0.275449
Enterprise    0.101796
Name: proportion, dtype: float64

Behaviour:
behaviour_at_churn
Declining            0.491018
Stable               0.299401
Rapidly Declining    0.179641
Improving            0.029940
Name: proportion, dtype: float64

Reasons:
churn_reason
Low Product Engagement           0.257485
Poor Customer Experience         0.179641
Product / Technical Issues       0.173653
Business / Budget Constraints    0.143713
Contract / Renewal Decision      0.131737
Lack of Required Features        0.113772
Name: proportion, dtype: float64

Dates:
2024-02-27 00:00:00
2025-12-25 00:00:00


In [190]:
import importlib
import src.churn

importlib.reload(src.churn)

generate_churn = src.churn.generate_churn

In [188]:
print(customer_state.columns.tolist())
print(usage.columns.tolist())
print(support.columns.tolist())

['customer_id', 'week_start', 'health_score', 'baseline_health', 'health_trend', 'volatility', 'behaviour']
['customer_id', 'week_start', 'active_users', 'sessions', 'active_days', 'usage_minutes', 'features_used', 'p001_sessions', 'p002_sessions', 'p003_sessions', 'p004_sessions', 'p005_sessions']
['ticket_id', 'customer_id', 'created_at', 'resolved_at', 'issue_category', 'priority', 'channel', 'resolution_hours', 'reopened', 'csat_score', 'escalated']


In [204]:
import importlib
import src.churn

importlib.reload(src.churn)

generate_churn = src.churn.generate_churn

In [199]:
test_customers = customers.sample(
    n=1200,
    random_state=42
).copy()

test_customer_ids = set(test_customers["customer_id"])

test_state = customer_state[
    customer_state["customer_id"].isin(test_customer_ids)
].copy()

test_usage = usage[
    usage["customer_id"].isin(test_customer_ids)
].copy()

test_support = support[
    support["customer_id"].isin(test_customer_ids)
].copy()

test_contracts = contracts[
    contracts["customer_id"].isin(test_customer_ids)
].copy()

In [200]:
from src.churn import _prepare_support_risk

test_state_dates = test_state[
    ["customer_id", "week_start"]
].copy()

test_support_risk = _prepare_support_risk(
    test_support,
    test_state_dates
)

print(test_support_risk.shape)
print(test_support_risk.columns.tolist())
print(test_support_risk.head())
print(test_support_risk.isna().sum())

(189600, 8)
['customer_id', 'week_start', 'support_ticket_count', 'support_escalation_rate', 'support_reopen_rate', 'support_avg_csat', 'support_avg_resolution_hours', 'support_risk_multiplier']
  customer_id week_start  support_ticket_count  support_escalation_rate  \
0      C00058 2023-01-02                   0.0                      0.0   
1      C00060 2023-01-02                   0.0                      0.0   
2      C00089 2023-01-02                   0.0                      0.0   
3      C00104 2023-01-02                   0.0                      0.0   
4      C00116 2023-01-02                   0.0                      0.0   

   support_reopen_rate  support_avg_csat  support_avg_resolution_hours  \
0                  0.0               4.0                          24.0   
1                  0.0               4.0                          24.0   
2                  0.0               4.0                          24.0   
3                  0.0               4.0                  

In [205]:
test_churn = generate_churn(
    customers=test_customers,
    contracts=test_contracts,
    customer_state=test_state,
    usage=test_usage,
    support=test_support,
    seed=42,
)

print("Shape:", test_churn.shape)
print(
    "Unique churned customers:",
    test_churn["customer_id"].nunique()
)

print(
    "Churn rate:",
    round(
        test_churn["customer_id"].nunique()
        / len(test_customers),
        3
    )
)

print("\nSegment:")
print(
    test_churn
    .merge(
        test_customers[
            ["customer_id", "segment"]
        ],
        on="customer_id",
        how="left",
    )["segment"]
    .value_counts(normalize=True)
)

print("\nBehaviour:")
print(
    test_churn[
        "behaviour_at_churn"
    ].value_counts(normalize=True)
)

print("\nReasons:")
print(
    test_churn[
        "churn_reason"
    ].value_counts(normalize=True)
)

print("\nHealth at churn:")
print(
    test_churn[
        "health_score_at_churn"
    ].describe()
)

print("\nChurn dates:")
print(
    test_churn["churn_date"].min(),
    "→",
    test_churn["churn_date"].max()
)

print("\nMissing values:")
print(
    test_churn.isna().sum()
)

print(
    "\nDuplicate customers:",
    test_churn["customer_id"].duplicated().sum()
)

Preparing usage risk...
Preparing support risk...
Simulating weekly churn...
Shape: (168, 7)
Unique churned customers: 168
Churn rate: 0.14

Segment:
segment
SMB           0.690476
Mid-Market    0.208333
Enterprise    0.101190
Name: proportion, dtype: float64

Behaviour:
behaviour_at_churn
Declining            0.410714
Stable               0.357143
Rapidly Declining    0.196429
Improving            0.035714
Name: proportion, dtype: float64

Reasons:
churn_reason
Poor Customer Experience         0.226190
Low Product Engagement           0.220238
Product / Technical Issues       0.178571
Business / Budget Constraints    0.178571
Lack of Required Features        0.113095
Contract / Renewal Decision      0.083333
Name: proportion, dtype: float64

Health at churn:
count    168.000000
mean      49.773036
std       25.999403
min        0.000000
25%       30.377500
50%       50.680000
75%       67.797500
max      100.000000
Name: health_score_at_churn, dtype: float64

Churn dates:
2024-01-29 0

In [206]:
segment_validation = (
    test_customers[
        ["customer_id", "segment"]
    ]
    .merge(
        test_churn[
            ["customer_id"]
        ].assign(churned=1),
        on="customer_id",
        how="left",
    )
)

segment_validation["churned"] = (
    segment_validation["churned"]
    .fillna(0)
)

print(
    segment_validation
    .groupby("segment")["churned"]
    .agg(
        customers="count",
        churned="sum",
        churn_rate="mean",
    )
)

            customers  churned  churn_rate
segment                                   
Enterprise        204     17.0    0.083333
Mid-Market        351     35.0    0.099715
SMB               645    116.0    0.179845


In [207]:
churn = generate_churn(
    customers=customers,
    contracts=contracts,
    customer_state=customer_state,
    usage=usage,
    support=support,
    seed=42,
)

Preparing usage risk...
Preparing support risk...
Simulating weekly churn...


In [209]:
churn.to_parquet(
    "../data/processed/churn.parquet",
    index=False,
)

churn.sample(
    min(1000, len(churn)),
    random_state=42,
).to_csv(
    "../data/samples/churn_sample.csv",
    index=False,
)

In [210]:
# =========================================================
# Final churn validation
# =========================================================

print("Shape:", churn.shape)
print("Unique customers:", churn["customer_id"].nunique())

print(
    "Overall churn rate:",
    round(
        churn["customer_id"].nunique() / len(customers),
        4
    )
)

print("\nChurn by segment:")

segment_check = (
    customers[
        ["customer_id", "segment"]
    ]
    .merge(
        churn[
            ["customer_id"]
        ].assign(churned=1),
        on="customer_id",
        how="left",
    )
)

segment_check["churned"] = (
    segment_check["churned"]
    .fillna(0)
)

print(
    segment_check
    .groupby("segment")["churned"]
    .agg(
        customers="count",
        churned="sum",
        churn_rate="mean",
    )
)

print("\nBehaviour at churn:")
print(
    churn["behaviour_at_churn"]
    .value_counts(normalize=True)
)

print("\nChurn reasons:")
print(
    churn["churn_reason"]
    .value_counts(normalize=True)
)

print("\nHealth at churn:")
print(
    churn["health_score_at_churn"]
    .describe()
)

print("\nChurn date range:")
print(
    churn["churn_date"].min(),
    "→",
    churn["churn_date"].max()
)

print("\nMissing values:")
print(
    churn.isna().sum()
)

print(
    "\nDuplicate customer churn events:",
    churn["customer_id"].duplicated().sum()
)

Shape: (2375, 7)
Unique customers: 2375
Overall churn rate: 0.1187

Churn by segment:
            customers  churned  churn_rate
segment                                   
Enterprise       2972    228.0    0.076716
Mid-Market       6083    623.0    0.102417
SMB             10945   1524.0    0.139242

Behaviour at churn:
behaviour_at_churn
Declining            0.464000
Stable               0.307789
Rapidly Declining    0.194105
Improving            0.034105
Name: proportion, dtype: float64

Churn reasons:
churn_reason
Low Product Engagement           0.259368
Poor Customer Experience         0.216421
Product / Technical Issues       0.168842
Contract / Renewal Decision      0.138105
Business / Budget Constraints    0.133895
Lack of Required Features        0.083368
Name: proportion, dtype: float64

Health at churn:
count    2375.000000
mean       47.322737
std        25.798314
min         0.000000
25%        28.915000
50%        46.660000
75%        66.360000
max       100.000000
Name: 

In [211]:
from pathlib import Path

churn_path = Path("../data/processed/churn.parquet")

print("Exists:", churn_path.exists())

if churn_path.exists():
    print(
        "Size:",
        round(churn_path.stat().st_size / (1024 ** 2), 2),
        "MB"
    )

Exists: True
Size: 0.06 MB


In [232]:
import importlib
import src.labels

importlib.reload(src.labels)

generate_churn_labels = src.labels.generate_churn_labels

In [213]:
print("Generating full 90-day churn labels...")

churn_labels = generate_churn_labels(
    customers=customers,
    customer_state=customer_state,
    churn=churn,
    horizon_days=90,
)

print("Done.")
print("Shape:", churn_labels.shape)

Generating full 90-day churn labels...
Done.
Shape: (2836830, 3)


In [214]:
print("\n==============================")
print("LABEL VALIDATION")
print("==============================")

print(
    "Shape:",
    churn_labels.shape
)

print(
    "Unique customers:",
    churn_labels["customer_id"].nunique()
)

print(
    "Snapshot range:",
    churn_labels["snapshot_date"].min(),
    "→",
    churn_labels["snapshot_date"].max()
)

print("\nLabel distribution:")

print(
    churn_labels["churn_90d"]
    .value_counts()
)

print("\nLabel proportions:")

print(
    churn_labels["churn_90d"]
    .value_counts(
        normalize=True
    )
)

print("\nMissing values:")

print(
    churn_labels.isna().sum()
)

print(
    "\nDuplicate customer-snapshot pairs:",
    churn_labels.duplicated(
        [
            "customer_id",
            "snapshot_date",
        ]
    ).sum()
)


LABEL VALIDATION
Shape: (2836830, 3)
Unique customers: 20000
Snapshot range: 2023-01-02 00:00:00 → 2025-10-06 00:00:00

Label distribution:
churn_90d
0    2809298
1      27532
Name: count, dtype: int64

Label proportions:
churn_90d
0    0.990295
1    0.009705
Name: proportion, dtype: float64

Missing values:
customer_id      0
snapshot_date    0
churn_90d        0
dtype: int64

Duplicate customer-snapshot pairs: 0


In [215]:
validation = churn_labels.merge(
    churn[
        [
            "customer_id",
            "churn_date",
        ]
    ],
    on="customer_id",
    how="inner",
)

post_churn_rows = (
    validation[
        validation["snapshot_date"]
        >= validation["churn_date"]
    ]
)

print(
    "Post-churn snapshots:",
    len(post_churn_rows)
)

Post-churn snapshots: 0


In [216]:
positive_customers = (
    churn_labels.loc[
        churn_labels["churn_90d"] == 1,
        "customer_id",
    ]
    .nunique()
)

print(
    "Churned customers:",
    churn["customer_id"].nunique()
)

print(
    "Customers with positive labels:",
    positive_customers
)

Churned customers: 2375
Customers with positive labels: 2336


In [217]:
churn_labels.to_parquet(
    "../data/processed/churn_labels.parquet",
    index=False,
)

churn_labels.sample(
    min(1000, len(churn_labels)),
    random_state=42,
).to_csv(
    "../data/samples/churn_labels_sample.csv",
    index=False,
)

print(
    "\nSaved:",
    "../data/processed/churn_labels.parquet"
)

print(
    "Sample:",
    "../data/samples/churn_labels_sample.csv"
)


Saved: ../data/processed/churn_labels.parquet
Sample: ../data/samples/churn_labels_sample.csv


In [219]:
from pathlib import Path

path = Path("../data/processed/churn_labels.parquet")

print("Exists:", path.exists())

if path.exists():
    print(
        "Size:",
        round(path.stat().st_size / (1024 ** 2), 2),
        "MB"
    )

Exists: True
Size: 0.39 MB


In [220]:
from src.features import (
    generate_features,
    validate_features,
)

In [222]:
import pandas as pd

customers = pd.read_csv(
    "../data/raw/customers.csv"
)

contracts = pd.read_csv(
    "../data/raw/contracts.csv"
)

subscriptions = pd.read_csv(
    "../data/raw/subscriptions.csv"
)

customer_state = pd.read_parquet(
    "../data/processed/customer_state.parquet"
)

usage = pd.read_parquet(
    "../data/processed/usage.parquet"
)

support = pd.read_parquet(
    "../data/processed/support.parquet"
)

churn_labels = pd.read_parquet(
    "../data/processed/churn_labels.parquet"
)

In [223]:
features = generate_features(
    customers=customers,
    contracts=contracts,
    subscriptions=subscriptions,
    customer_state=customer_state,
    usage=usage,
    support=support,
    churn_labels=churn_labels,
)

In [224]:
validate_features(
    features,
    churn_labels,
)

Feature validation passed.
Rows: 2,836,830
Customers: 20,000
Snapshots: 145
Positive labels: 27,532
Positive rate: 0.9705%


In [225]:
features.to_parquet(
    "../data/processed/features.parquet",
    index=False,
)

features.sample(
    min(1000, len(features)),
    random_state=42,
).to_csv(
    "../data/samples/features_sample.csv",
    index=False,
)

In [227]:
print(features.shape)
features.columns.tolist()

(2836830, 101)


['customer_id',
 'snapshot_date',
 'churn_90d',
 'industry',
 'region',
 'segment',
 'company_size',
 'acquisition_channel',
 'tenure_days',
 'tenure_years',
 'health_score',
 'baseline_health',
 'health_trend',
 'volatility',
 'behaviour',
 'health_score_change_4w',
 'health_score_change_8w',
 'health_score_change_12w',
 'health_score_pct_change_4w',
 'health_score_pct_change_8w',
 'health_score_pct_change_12w',
 'active_users',
 'sessions',
 'active_days',
 'usage_minutes',
 'features_used',
 'p001_sessions',
 'p002_sessions',
 'p003_sessions',
 'p004_sessions',
 'p005_sessions',
 'active_users_change_4w',
 'active_users_change_8w',
 'active_users_change_12w',
 'sessions_change_4w',
 'sessions_change_8w',
 'sessions_change_12w',
 'usage_minutes_change_4w',
 'usage_minutes_change_8w',
 'usage_minutes_change_12w',
 'active_days_change_4w',
 'active_days_change_8w',
 'active_days_change_12w',
 'features_used_change_4w',
 'features_used_change_8w',
 'features_used_change_12w',
 'active_u

In [228]:
features.isna().mean().sort_values(
    ascending=False
).head(20)

avg_resolution_hours_4w         0.929935
avg_csat_4w                     0.929935
avg_csat_8w                     0.900485
avg_resolution_hours_8w         0.900485
sessions_pct_change_12w         0.886057
usage_minutes_pct_change_12w    0.886057
features_used_pct_change_12w    0.886057
active_days_pct_change_12w      0.885942
active_users_pct_change_12w     0.885872
avg_csat_12w                    0.885225
avg_resolution_hours_12w        0.885225
usage_minutes_pct_change_8w     0.874802
sessions_pct_change_8w          0.874802
features_used_pct_change_8w     0.874802
active_days_pct_change_8w       0.874665
active_users_pct_change_8w      0.874584
sessions_pct_change_4w          0.863037
features_used_pct_change_4w     0.863037
usage_minutes_pct_change_4w     0.863037
active_days_pct_change_4w       0.862876
dtype: float64

In [229]:
print(features["churn_90d"].value_counts())
print(features["churn_90d"].mean())

churn_90d
0    2809298
1      27532
Name: count, dtype: int64
0.009705199113094545


In [230]:
features[
    [
        "health_score",
        "active_users",
        "sessions",
        "usage_minutes",
        "product_count",
        "tickets_12w",
        "avg_csat_12w",
        "days_to_renewal",
        "annual_contract_value",
        "tenure_days",
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
health_score,2836830.0,7.230371e+01,1.820935e+01,0.0,62.460,73.85,84.850,100.0
active_users,2836830.0,4.602770e+01,3.399873e+02,0.0,0.000,0.00,0.000,20332.0
sessions,2836830.0,3.369490e+02,2.771630e+03,0.0,0.000,0.00,0.000,181133.0
usage_minutes,2836830.0,6.483028e+03,5.349750e+04,0.0,0.000,0.00,0.000,3817605.0
product_count,2836830.0,2.466014e+00,1.092915e+00,1.0,2.000,2.00,3.000,5.0
tickets_12w,2836830.0,2.983922e-01,1.058077e+00,0.0,0.000,0.00,0.000,22.0
avg_csat_12w,325597.0,4.067901e+00,5.240816e-01,2.0,3.875,4.00,4.375,5.0
days_to_renewal,424607.0,3.640536e+02,2.539023e+02,0.0,186.000,300.00,524.000,1095.0
annual_contract_value,424607.0,4.475031e+06,8.555697e+06,200000.0,489434.120,1178941.01,4457137.310,100000000.0
tenure_days,2836830.0,-4.253197e+02,4.211307e+02,-1459.0,-727.000,-421.00,-124.000,644.0


In [233]:
from src.labels import generate_churn_labels

churn_labels = generate_churn_labels(
    customers=customers,
    churn=churn,
    customer_state=customer_state,
    contracts=contracts,
)

In [234]:
print(churn_labels.shape)
print(churn_labels.head())
print(churn_labels.tail())

(424607, 3)
  customer_id snapshot_date  churn_90d
0      C00001    2024-09-23          0
1      C00001    2024-09-30          0
2      C00001    2024-10-07          0
3      C00001    2024-10-14          0
4      C00001    2024-10-21          0
       customer_id snapshot_date  churn_90d
424602      C19999    2025-09-08          0
424603      C19999    2025-09-15          0
424604      C19999    2025-09-22          0
424605      C19999    2025-09-29          0
424606      C19999    2025-10-06          0


In [235]:
print("Shape:", churn_labels.shape)
print(
    "Unique customers:",
    churn_labels["customer_id"].nunique()
)
print(
    "Unique snapshots:",
    churn_labels["snapshot_date"].nunique()
)

Shape: (424607, 3)
Unique customers: 11716
Unique snapshots: 93


In [236]:
print(
    churn_labels["churn_90d"]
    .value_counts()
)

print(
    "Positive rate:",
    churn_labels["churn_90d"].mean()
)

churn_90d
0    401578
1     23029
Name: count, dtype: int64
Positive rate: 0.05423603473329455


In [237]:
duplicates = churn_labels.duplicated(
    ["customer_id", "snapshot_date"]
).sum()

print(
    "Duplicate customer-snapshot pairs:",
    duplicates
)

Duplicate customer-snapshot pairs: 0


In [238]:
check = churn_labels.merge(
    customers[
        [
            "customer_id",
            "signup_date",
        ]
    ],
    on="customer_id",
    how="left",
)

check["signup_date"] = pd.to_datetime(
    check["signup_date"]
)

invalid_signup = (
    check["snapshot_date"]
    < check["signup_date"]
)

print(
    "Snapshots before signup:",
    invalid_signup.sum()
)

Snapshots before signup: 0


In [239]:
contract_check = churn_labels.merge(
    contracts[
        [
            "customer_id",
            "start_date",
            "end_date",
        ]
    ],
    on="customer_id",
    how="left",
)

contract_check["start_date"] = pd.to_datetime(
    contract_check["start_date"]
)

contract_check["end_date"] = pd.to_datetime(
    contract_check["end_date"]
)

valid_contract = (
    (
        contract_check["snapshot_date"]
        >= contract_check["start_date"]
    )
    &
    (
        contract_check["snapshot_date"]
        <= contract_check["end_date"]
    )
)

print(
    "Snapshots outside contract lifecycle:",
    (~valid_contract).sum()
)

Snapshots outside contract lifecycle: 0


In [243]:
churn_labels.to_parquet(
    "../data/processed/churn_labels.parquet",
    index=False,
)

churn_labels.sample(
    min(1000, len(churn_labels)),
    random_state=42,
).to_csv(
    "../data/samples/churn_labels_sample.csv",
    index=False,
)

In [244]:
print(
    churn_labels.groupby("snapshot_date")
    .size()
    .describe()
)

count      93.000000
mean     4565.666667
std      2201.057648
min        13.000000
25%      2765.000000
50%      5309.000000
75%      6436.000000
max      7106.000000
dtype: float64


In [245]:
print(
    churn_labels.groupby("snapshot_date")
    .size()
    .head(10)
)

print(
    churn_labels.groupby("snapshot_date")
    .size()
    .tail(10)
)

snapshot_date
2024-01-01      13
2024-01-08     132
2024-01-15     242
2024-01-22     353
2024-01-29     484
2024-02-05     612
2024-02-12     729
2024-02-19     853
2024-02-26     967
2024-03-04    1082
dtype: int64
snapshot_date
2025-08-04    6885
2025-08-11    6889
2025-08-18    6902
2025-08-25    6931
2025-09-01    6961
2025-09-08    6993
2025-09-15    7060
2025-09-22    7057
2025-09-29    7073
2025-10-06    7106
dtype: int64


In [246]:
customer_eligibility = (
    churn_labels.groupby("customer_id")
    .agg(
        first_snapshot=("snapshot_date", "min"),
        last_snapshot=("snapshot_date", "max"),
        snapshot_count=("snapshot_date", "count"),
    )
)

print(customer_eligibility.describe())

                   first_snapshot               last_snapshot  snapshot_count
count                       11716                       11716    11716.000000
mean   2024-11-22 19:49:08.583134  2025-07-27 12:24:49.655172       36.241635
min           2024-01-01 00:00:00         2024-01-15 00:00:00        1.000000
25%           2024-06-17 00:00:00         2025-06-09 00:00:00       18.000000
50%           2024-11-25 00:00:00         2025-10-06 00:00:00       38.000000
75%           2025-05-05 00:00:00         2025-10-06 00:00:00       52.000000
max           2025-10-06 00:00:00         2025-10-06 00:00:00       93.000000
std                           NaN                         NaN       21.169827


In [258]:
import importlib
import src.labels

importlib.reload(src.features)

generate_features = src.features.generate_features

In [259]:
features = generate_features(
    customers=customers,
    contracts=contracts,
    subscriptions=subscriptions,
    customer_state=customer_state,
    usage=usage,
    support=support,
    churn_labels=churn_labels,
)

In [260]:
validate_features(
    features,
    churn_labels,
)

Feature validation passed.
Rows: 424,607
Customers: 11,716
Snapshots: 93
Positive labels: 23,029
Positive rate: 5.4236%


In [261]:
print("Shape:", features.shape)

print("\nColumns:")
print(features.columns.tolist())

print("\nProduct count:")
print(features["product_count"].describe())

print("\nProduct count by snapshot:")
print(
    features.groupby("snapshot_date")["product_count"]
    .mean()
    .head(10)
)

print("\nMissingness:")
print(
    features.isna()
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

print("\nTarget:")
print(
    features["churn_90d"]
    .value_counts()
)

print(
    "Positive rate:",
    features["churn_90d"].mean()
)

print("\nTenure:")
print(
    features["tenure_days"].describe()
)

print("\nContract age:")
print(
    features["contract_age_days"].describe()
)

print("\nDays to renewal:")
print(
    features["days_to_renewal"].describe()
)

Shape: (424607, 101)

Columns:
['customer_id', 'snapshot_date', 'churn_90d', 'industry', 'region', 'segment', 'company_size', 'acquisition_channel', 'tenure_days', 'tenure_years', 'health_score', 'baseline_health', 'health_trend', 'volatility', 'behaviour', 'health_score_change_4w', 'health_score_change_8w', 'health_score_change_12w', 'health_score_pct_change_4w', 'health_score_pct_change_8w', 'health_score_pct_change_12w', 'active_users', 'sessions', 'active_days', 'usage_minutes', 'features_used', 'p001_sessions', 'p002_sessions', 'p003_sessions', 'p004_sessions', 'p005_sessions', 'active_users_change_4w', 'active_users_change_8w', 'active_users_change_12w', 'sessions_change_4w', 'sessions_change_8w', 'sessions_change_12w', 'usage_minutes_change_4w', 'usage_minutes_change_8w', 'usage_minutes_change_12w', 'active_days_change_4w', 'active_days_change_8w', 'active_days_change_12w', 'features_used_change_4w', 'features_used_change_8w', 'features_used_change_12w', 'active_users_pct_change

In [262]:
# ------------------------------------------------------------
# Temporal sanity check
# ------------------------------------------------------------

usage_check = usage.copy()
usage_check["week_start"] = pd.to_datetime(
    usage_check["week_start"]
)

support_check = support.copy()
support_check["created_at"] = pd.to_datetime(
    support_check["created_at"]
)

support_check["week_start"] = (
    support_check["created_at"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

print(
    "Earliest usage week:",
    usage_check["week_start"].min()
)

print(
    "Latest usage week:",
    usage_check["week_start"].max()
)

print(
    "Earliest support week:",
    support_check["week_start"].min()
)

print(
    "Latest support week:",
    support_check["week_start"].max()
)

print(
    "Earliest feature snapshot:",
    features["snapshot_date"].min()
)

print(
    "Latest feature snapshot:",
    features["snapshot_date"].max()
)

Earliest usage week: 2023-01-02 00:00:00
Latest usage week: 2026-01-05 00:00:00
Earliest support week: 2024-01-01 00:00:00
Latest support week: 2026-01-05 00:00:00
Earliest feature snapshot: 2024-01-01 00:00:00
Latest feature snapshot: 2025-10-06 00:00:00


In [249]:
print("Shape:", features.shape)
print(
    "Customers:",
    features["customer_id"].nunique()
)
print(
    "Snapshots:",
    features["snapshot_date"].nunique()
)

Shape: (424607, 101)
Customers: 11716
Snapshots: 93


In [250]:
features[
    [
        "tenure_days",
        "tenure_years",
        "contract_age_days",
        "days_to_renewal",
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
tenure_days,424607.0,169.634460,123.248328,0.0,68.000000,149.00000,253.000000,644.000000
tenure_years,424607.0,0.464434,0.337436,0.0,0.186174,0.40794,0.692676,1.763176
contract_age_days,424607.0,169.634460,123.248328,0.0,68.000000,149.00000,253.000000,644.000000
days_to_renewal,424607.0,364.053621,253.902279,0.0,186.000000,300.00000,524.000000,1095.000000


In [251]:
missing = (
    features.isna()
    .mean()
    .sort_values(ascending=False)
)

print(missing.head(25))

avg_resolution_hours_4w         0.536845
avg_csat_4w                     0.536845
avg_csat_8w                     0.352818
avg_resolution_hours_8w         0.352818
sessions_pct_change_12w         0.304642
usage_minutes_pct_change_12w    0.304642
features_used_pct_change_12w    0.304642
active_days_pct_change_12w      0.303914
active_users_pct_change_12w     0.303485
avg_csat_12w                    0.266757
avg_resolution_hours_12w        0.266757
usage_minutes_pct_change_8w     0.210077
sessions_pct_change_8w          0.210077
features_used_pct_change_8w     0.210077
active_days_pct_change_8w       0.209184
active_users_pct_change_8w      0.208685
sessions_pct_change_4w          0.109492
features_used_pct_change_4w     0.109492
usage_minutes_pct_change_4w     0.109492
active_days_pct_change_4w       0.108430
active_users_pct_change_4w      0.107822
health_score_pct_change_4w      0.002292
health_score_pct_change_8w      0.001861
health_score_pct_change_12w     0.001439
churn_90d       

In [252]:
print(
    features["churn_90d"].value_counts()
)

print(
    "Churn rate:",
    f"{features['churn_90d'].mean():.4%}"
)

churn_90d
0    401578
1     23029
Name: count, dtype: int64
Churn rate: 5.4236%


In [253]:
import pandas as pd
import numpy as np

print("=" * 70)
print("RETAIN-AI FEATURE QUALITY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Shape
# ------------------------------------------------------------

print("\n[1] DATASET SHAPE")

print(f"Rows:       {len(features):,}")
print(
    f"Customers:  "
    f"{features['customer_id'].nunique():,}"
)
print(
    f"Snapshots:  "
    f"{features['snapshot_date'].nunique():,}"
)
print(
    f"Features:   "
    f"{features.shape[1]:,}"
)

# ------------------------------------------------------------
# 2. Duplicate keys
# ------------------------------------------------------------

print("\n[2] DUPLICATE CHECK")

duplicates = features.duplicated(
    ["customer_id", "snapshot_date"]
).sum()

print(
    f"Duplicate customer-snapshot rows: "
    f"{duplicates:,}"
)

# ------------------------------------------------------------
# 3. Target
# ------------------------------------------------------------

print("\n[3] TARGET DISTRIBUTION")

target_counts = (
    features["churn_90d"]
    .value_counts()
    .sort_index()
)

print(target_counts)

print(
    f"Churn rate: "
    f"{features['churn_90d'].mean():.4%}"
)

# ------------------------------------------------------------
# 4. Missingness
# ------------------------------------------------------------

print("\n[4] MISSINGNESS")

missing = (
    features.isna()
    .mean()
    .sort_values(ascending=False)
)

missing_table = pd.DataFrame({
    "missing_count": features.isna().sum(),
    "missing_pct": missing,
})

print(
    missing_table[
        missing_table["missing_count"] > 0
    ].head(30)
)

# ------------------------------------------------------------
# 5. Data types
# ------------------------------------------------------------

print("\n[5] DATA TYPES")

print(
    features.dtypes.value_counts()
)

# ------------------------------------------------------------
# 6. Cardinality
# ------------------------------------------------------------

print("\n[6] CATEGORICAL CARDINALITY")

categorical_columns = features.select_dtypes(
    include=["object", "category", "bool"]
).columns

for column in categorical_columns:
    print(
        f"{column:35s}"
        f"{features[column].nunique(dropna=False):>8,}"
    )

# ------------------------------------------------------------
# 7. Numeric descriptive statistics
# ------------------------------------------------------------

print("\n[7] NUMERIC SUMMARY")

numeric_columns = features.select_dtypes(
    include=np.number
).columns

numeric_summary = (
    features[numeric_columns]
    .describe()
    .T
)

print(
    numeric_summary[
        ["count", "mean", "std", "min", "max"]
    ].to_string()
)

# ------------------------------------------------------------
# 8. Constant features
# ------------------------------------------------------------

print("\n[8] CONSTANT FEATURES")

constant_features = [
    column
    for column in features.columns
    if features[column].nunique(
        dropna=False
    ) <= 1
]

print(constant_features)

# ------------------------------------------------------------
# 9. Near-constant features
# ------------------------------------------------------------

print("\n[9] NEAR-CONSTANT NUMERIC FEATURES")

near_constant = []

for column in numeric_columns:

    if column == "churn_90d":
        continue

    value_counts = (
        features[column]
        .value_counts(
            normalize=True,
            dropna=False
        )
    )

    if len(value_counts) > 0:
        top_share = value_counts.iloc[0]

        if top_share >= 0.995:
            near_constant.append(
                (column, top_share)
            )

for column, share in near_constant:
    print(
        f"{column:45s}"
        f"{share:.4%}"
    )

# ------------------------------------------------------------
# 10. Target rate by behaviour
# ------------------------------------------------------------

print("\n[10] CHURN RATE BY BEHAVIOUR")

print(
    features.groupby("behaviour")[
        "churn_90d"
    ].agg(
        customers="count",
        churns="sum",
        churn_rate="mean",
    ).sort_values(
        "churn_rate",
        ascending=False
    )
)

# ------------------------------------------------------------
# 11. Target rate by segment
# ------------------------------------------------------------

print("\n[11] CHURN RATE BY SEGMENT")

print(
    features.groupby("segment")[
        "churn_90d"
    ].agg(
        customers="count",
        churns="sum",
        churn_rate="mean",
    ).sort_values(
        "churn_rate",
        ascending=False
    )
)

# ------------------------------------------------------------
# 12. Target rate by support burden
# ------------------------------------------------------------

print("\n[12] CHURN RATE BY SUPPORT BURDEN")

print(
    features.groupby(
        "high_support_burden_12w"
    )["churn_90d"].agg(
        customers="count",
        churns="sum",
        churn_rate="mean",
    )
)

# ------------------------------------------------------------
# 13. Target rate by usage deterioration
# ------------------------------------------------------------

print("\n[13] CHURN RATE BY USAGE DECLINE")

print(
    features.groupby(
        "usage_decline_12w_flag"
    )["churn_90d"].agg(
        customers="count",
        churns="sum",
        churn_rate="mean",
    )
)

print("\n" + "=" * 70)
print("AUDIT COMPLETE")
print("=" * 70)

RETAIN-AI FEATURE QUALITY AUDIT

[1] DATASET SHAPE
Rows:       424,607
Customers:  11,716
Snapshots:  93
Features:   101

[2] DUPLICATE CHECK
Duplicate customer-snapshot rows: 0

[3] TARGET DISTRIBUTION
churn_90d
0    401578
1     23029
Name: count, dtype: int64
Churn rate: 5.4236%

[4] MISSINGNESS
                              missing_count  missing_pct
active_days_pct_change_12w           129044     0.303914
active_days_pct_change_4w             46040     0.108430
active_days_pct_change_8w             88821     0.209184
active_users_pct_change_12w          128862     0.303485
active_users_pct_change_4w            45782     0.107822
active_users_pct_change_8w            88609     0.208685
avg_csat_12w                         113267     0.266757
avg_csat_4w                          227948     0.536845
avg_csat_8w                          149809     0.352818
avg_resolution_hours_12w             113267     0.266757
avg_resolution_hours_4w              227948     0.536845
avg_resolution_h

C:\Users\yadav\AppData\Local\Temp\ipykernel_10092\1609512250.py:101: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = features.select_dtypes(


                                  count          mean           std          min           max
churn_90d                      424607.0  5.423603e-02  2.264831e-01       0.0000  1.000000e+00
company_size                   424607.0  1.511828e+03  3.504618e+03      10.0000  5.000000e+04
tenure_days                    424607.0  1.696345e+02  1.232483e+02       0.0000  6.440000e+02
tenure_years                   424607.0  4.644338e-01  3.374355e-01       0.0000  1.763176e+00
health_score                   424607.0  7.205468e+01  2.102982e+01       0.0000  1.000000e+02
baseline_health                424607.0  7.533466e+01  9.887594e+00      40.0000  1.000000e+02
health_trend                   424607.0 -1.161539e-02  1.878339e-01      -0.7865  4.778000e-01
volatility                     424607.0  8.009929e-01  2.287302e-01       0.4001  1.200000e+00
health_score_change_4w         424607.0 -2.029394e-01  1.688698e+00     -10.2000  8.880000e+00
health_score_change_8w         424607.0 -4.019017e

In [254]:
print(subscriptions.columns.tolist())
print(subscriptions.head())
print(subscriptions.dtypes)

['subscription_id', 'customer_id', 'product_id', 'start_date', 'end_date', 'annual_value', 'monthly_value', 'status']
  subscription_id customer_id product_id  start_date    end_date  \
0      SUB0000001      C00001       P001  2024-09-19  2027-09-19   
1      SUB0000002      C00001       P003  2024-09-19  2027-09-19   
2      SUB0000003      C00002       P001  2025-05-22  2027-05-22   
3      SUB0000004      C00003       P001  2024-09-24  2026-09-24   
4      SUB0000005      C00003       P002  2024-09-24  2026-09-24   

   annual_value  monthly_value  status  
0     521883.05       43490.25  Active  
1     478116.95       39843.08  Active  
2     386859.14       32238.26  Active  
3    1855247.59      154603.97  Active  
4    4062363.35      338530.28  Active  
subscription_id        str
customer_id            str
product_id             str
start_date             str
end_date               str
annual_value       float64
monthly_value      float64
status                 str
dtype: obje

In [255]:
print(
    subscriptions.groupby("customer_id")
    .size()
    .describe()
)

count    20000.000000
mean         2.462950
std          1.092377
min          1.000000
25%          2.000000
50%          2.000000
75%          3.000000
max          5.000000
dtype: float64


In [256]:
history_check = features.groupby(
    "customer_id"
).agg(
    snapshot_count=("snapshot_date", "count"),
    first_snapshot=("snapshot_date", "min"),
    last_snapshot=("snapshot_date", "max"),
)

print(history_check["snapshot_count"].describe())

count    11716.000000
mean        36.241635
std         21.169827
min          1.000000
25%         18.000000
50%         38.000000
75%         52.000000
max         93.000000
Name: snapshot_count, dtype: float64


In [257]:
print(
    (
        features["health_score_change_12w"].notna()
    ).mean()
)

print(
    (
        features["usage_minutes_pct_change_12w"].notna()
    ).mean()
)

1.0
0.6953582960243236


In [ ]:
import importlib
import src.labels

importlib.reload(src.labels)

generate_churn_labels = src.labels.generate_churn_labels

In [263]:
import numpy as np
import pandas as pd


# ============================================================
# 1. BASIC STRUCTURE
# ============================================================

print("=" * 70)
print("1. BASIC STRUCTURE")
print("=" * 70)

print("Shape:", features.shape)

print(
    "Duplicate customer-snapshot rows:",
    features[
        ["customer_id", "snapshot_date"]
    ].duplicated().sum()
)

print(
    "Duplicate customer IDs:",
    features["customer_id"].duplicated().sum()
)

print(
    "Unique customers:",
    features["customer_id"].nunique()
)

print(
    "Unique snapshots:",
    features["snapshot_date"].nunique()
)


# ============================================================
# 2. TARGET
# ============================================================

print("\n" + "=" * 70)
print("2. TARGET")
print("=" * 70)

print(
    features["churn_90d"]
    .value_counts()
)

print(
    "\nPositive rate:",
    f"{features['churn_90d'].mean():.4%}"
)


# ============================================================
# 3. MISSINGNESS
# ============================================================

print("\n" + "=" * 70)
print("3. MISSINGNESS")
print("=" * 70)

missing = (
    features
    .isna()
    .mean()
    .sort_values(
        ascending=False
    )
)

print(
    missing[
        missing > 0
    ].head(30)
)


# ============================================================
# 4. CONSTANT FEATURES
# ============================================================

print("\n" + "=" * 70)
print("4. CONSTANT FEATURES")
print("=" * 70)

constant_features = [
    column
    for column in features.columns
    if features[column].nunique(
        dropna=False
    ) <= 1
]

print(
    constant_features
)


# ============================================================
# 5. ID-LIKE FEATURES
# ============================================================

print("\n" + "=" * 70)
print("5. ID-LIKE FEATURES")
print("=" * 70)

for column in [
    "customer_id",
    "contract_id",
]:

    print(
        f"{column}:",
        features[column].nunique(),
        "unique values"
    )


# ============================================================
# 6. EXTREME PERCENTAGE CHANGES
# ============================================================

print("\n" + "=" * 70)
print("6. EXTREME PERCENTAGE CHANGES")
print("=" * 70)

pct_columns = [
    column
    for column in features.columns
    if "_pct_change_" in column
]

for column in pct_columns:

    series = features[column].dropna()

    print(
        f"\n{column}"
    )

    print(
        series.describe(
            percentiles=[
                0.01,
                0.05,
                0.25,
                0.50,
                0.75,
                0.95,
                0.99,
            ]
        )
    )


# ============================================================
# 7. HEALTH / USAGE / SUPPORT DISTRIBUTIONS
# ============================================================

print("\n" + "=" * 70)
print("7. KEY NUMERIC FEATURES")
print("=" * 70)

key_numeric = [
    "health_score",
    "health_trend",
    "volatility",
    "active_users",
    "sessions",
    "active_days",
    "usage_minutes",
    "features_used",
    "product_count",
    "tickets_12w",
    "escalations_12w",
    "reopens_12w",
    "avg_csat_12w",
    "avg_resolution_hours_12w",
    "annual_contract_value",
    "days_to_renewal",
]

for column in key_numeric:

    if column not in features.columns:
        continue

    print(
        f"\n{column}"
    )

    print(
        features[column].describe()
    )


# ============================================================
# 8. HIGH CORRELATION NUMERIC FEATURES
# ============================================================

print("\n" + "=" * 70)
print("8. HIGH CORRELATION FEATURES")
print("=" * 70)

numeric_features = features.select_dtypes(
    include=np.number
).drop(
    columns=["churn_90d"],
    errors="ignore"
)

correlation_matrix = (
    numeric_features
    .corr()
    .abs()
)

upper = correlation_matrix.where(
    np.triu(
        np.ones(
            correlation_matrix.shape
        ),
        k=1
    ).astype(bool)
)

high_correlations = (
    upper
    .stack()
    .sort_values(
        ascending=False
    )
)

print(
    high_correlations[
        high_correlations >= 0.90
    ].head(50)
)


# ============================================================
# 9. TARGET RATE BY IMPORTANT FEATURES
# ============================================================

print("\n" + "=" * 70)
print("9. TARGET RELATIONSHIPS")
print("=" * 70)


print("\nBehaviour:")
print(
    features
    .groupby("behaviour")["churn_90d"]
    .agg(
        count="size",
        churn_rate="mean"
    )
    .sort_values(
        "churn_rate",
        ascending=False
    )
)


print("\nSegment:")
print(
    features
    .groupby("segment")["churn_90d"]
    .agg(
        count="size",
        churn_rate="mean"
    )
    .sort_values(
        "churn_rate",
        ascending=False
    )
)


print("\nProduct count:")
print(
    features
    .groupby("product_count")["churn_90d"]
    .agg(
        count="size",
        churn_rate="mean"
    )
)


# ============================================================
# 10. TEMPORAL COVERAGE
# ============================================================

print("\n" + "=" * 70)
print("10. TEMPORAL COVERAGE")
print("=" * 70)

print(
    "Feature start:",
    features["snapshot_date"].min()
)

print(
    "Feature end:",
    features["snapshot_date"].max()
)

print(
    "\nRows by snapshot:"
)

print(
    features
    .groupby("snapshot_date")
    .size()
    .head(15)
)

print(
    "\nLast snapshots:"
)

print(
    features
    .groupby("snapshot_date")
    .size()
    .tail(15)
)


# ============================================================
# 11. CUSTOMER HISTORY
# ============================================================

print("\n" + "=" * 70)
print("11. CUSTOMER HISTORY")
print("=" * 70)

history = (
    features
    .groupby("customer_id")
    .agg(
        snapshot_count=(
            "snapshot_date",
            "count"
        ),
        first_snapshot=(
            "snapshot_date",
            "min"
        ),
        last_snapshot=(
            "snapshot_date",
            "max"
        ),
    )
)

print(
    history["snapshot_count"].describe()
)

print(
    "\nCustomers with <12 snapshots:",
    (
        history["snapshot_count"] < 12
    ).sum()
)

print(
    "Customers with >=26 snapshots:",
    (
        history["snapshot_count"] >= 26
    ).sum()
)

print(
    "Customers with >=52 snapshots:",
    (
        history["snapshot_count"] >= 52
    ).sum()
)

1. BASIC STRUCTURE
Shape: (424607, 101)
Duplicate customer-snapshot rows: 0
Duplicate customer IDs: 412891
Unique customers: 11716
Unique snapshots: 93

2. TARGET
churn_90d
0    401578
1     23029
Name: count, dtype: int64

Positive rate: 5.4236%

3. MISSINGNESS
avg_resolution_hours_4w         0.550043
avg_csat_4w                     0.550043
avg_csat_8w                     0.371496
avg_resolution_hours_8w         0.371496
sessions_pct_change_12w         0.327331
usage_minutes_pct_change_12w    0.327331
features_used_pct_change_12w    0.327331
active_days_pct_change_12w      0.326650
active_users_pct_change_12w     0.326236
avg_csat_12w                    0.288219
avg_resolution_hours_12w        0.288219
usage_minutes_pct_change_8w     0.234247
sessions_pct_change_8w          0.234247
features_used_pct_change_8w     0.234247
active_days_pct_change_8w       0.233388
active_users_pct_change_8w      0.232914
sessions_pct_change_4w          0.135273
features_used_pct_change_4w     0.135273

In [264]:
# ============================================================
# Point-in-time product adoption examples
# ============================================================

subscription_check = subscriptions.copy()

subscription_check["start_date"] = pd.to_datetime(
    subscription_check["start_date"]
)

subscription_check["end_date"] = pd.to_datetime(
    subscription_check["end_date"]
)

sample_customer = (
    subscription_check["customer_id"]
    .value_counts()
    .index[0]
)

print("Sample customer:", sample_customer)

print(
    subscription_check[
        subscription_check["customer_id"]
        == sample_customer
    ][
        [
            "customer_id",
            "product_id",
            "start_date",
            "end_date",
        ]
    ].sort_values("start_date")
)

print("\nFeature product history:")

print(
    features[
        features["customer_id"]
        == sample_customer
    ][
        [
            "snapshot_date",
            "product_count",
        ]
    ].head(20)
)

Sample customer: C00023
   customer_id product_id start_date   end_date
54      C00023       P001 2026-07-21 2027-07-21
55      C00023       P002 2026-07-21 2027-07-21
56      C00023       P003 2026-07-21 2027-07-21
57      C00023       P004 2026-07-21 2027-07-21
58      C00023       P005 2026-07-21 2027-07-21

Feature product history:
Empty DataFrame
Columns: [snapshot_date, product_count]
Index: []


In [265]:
# Pick a customer that actually exists in the feature table
sample_customer = features["customer_id"].iloc[0]

print("Sample customer:", sample_customer)

print("\nSubscriptions:")
print(
    subscriptions[
        subscriptions["customer_id"] == sample_customer
    ][
        [
            "customer_id",
            "product_id",
            "start_date",
            "end_date",
        ]
    ].sort_values("start_date")
)

print("\nFeature product history:")
print(
    features[
        features["customer_id"] == sample_customer
    ][
        [
            "snapshot_date",
            "product_count",
        ]
    ].head(20)
)

Sample customer: C00001

Subscriptions:
  customer_id product_id  start_date    end_date
0      C00001       P001  2024-09-19  2027-09-19
1      C00001       P003  2024-09-19  2027-09-19

Feature product history:
   snapshot_date  product_count
0     2024-09-23              2
1     2024-09-30              2
2     2024-10-07              2
3     2024-10-14              2
4     2024-10-21              2
5     2024-10-28              2
6     2024-11-04              2
7     2024-11-11              2
8     2024-11-18              2
9     2024-11-25              2
10    2024-12-02              2
11    2024-12-09              2
12    2024-12-16              2
13    2024-12-23              2
14    2024-12-30              2
15    2025-01-06              2
16    2025-01-13              2
17    2025-01-20              2
18    2025-01-27              2
19    2025-02-03              2


In [267]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")

features_path = DATA_DIR / "processed" / "features.parquet"
sample_path = DATA_DIR / "samples" / "features_sample.csv"

print("Features file exists:", features_path.exists())
print("Sample file exists:", sample_path.exists())

if features_path.exists():
    saved_features = pd.read_parquet(features_path)

    print("\nSaved feature store:")
    print("Path:", features_path)
    print("Shape:", saved_features.shape)
    print("Customers:", saved_features["customer_id"].nunique())
    print("Snapshots:", saved_features["snapshot_date"].nunique())
    print("Positive labels:", saved_features["churn_90d"].sum())
    print("Positive rate:", f"{saved_features['churn_90d'].mean():.4%}")

Features file exists: True
Sample file exists: True

Saved feature store:
Path: ..\data\processed\features.parquet
Shape: (2836830, 101)
Customers: 20000
Snapshots: 145
Positive labels: 27532
Positive rate: 0.9705%


In [268]:
from pathlib import Path
import pandas as pd

# ============================================================
# Save the CURRENT validated feature dataframe
# ============================================================

DATA_DIR = Path("../data")

output_path = (
    DATA_DIR
    / "processed"
    / "features.parquet"
)

sample_path = (
    DATA_DIR
    / "samples"
    / "features_sample.csv"
)

# ------------------------------------------------------------
# Safety checks BEFORE saving
# ------------------------------------------------------------

assert features.shape == (424607, 101), (
    f"Unexpected feature shape: {features.shape}"
)

assert features["customer_id"].nunique() == 11716, (
    "Unexpected customer count."
)

assert features["snapshot_date"].nunique() == 93, (
    "Unexpected snapshot count."
)

assert features["churn_90d"].sum() == 23029, (
    "Unexpected positive-label count."
)

assert abs(
    features["churn_90d"].mean() - 0.05423603473329455
) < 1e-10, (
    "Unexpected target rate."
)

assert not features[
    ["customer_id", "snapshot_date"]
].duplicated().any(), (
    "Duplicate customer-snapshot pairs detected."
)

# ------------------------------------------------------------
# Save canonical feature store
# ------------------------------------------------------------

features.to_parquet(
    output_path,
    index=False,
)

# Save sample
features.sample(
    min(1000, len(features)),
    random_state=42,
).to_csv(
    sample_path,
    index=False,
)

print("Feature store saved successfully.")
print("Path:", output_path)
print("Shape:", features.shape)
print("Customers:", features["customer_id"].nunique())
print("Snapshots:", features["snapshot_date"].nunique())
print("Positive labels:", features["churn_90d"].sum())
print(
    "Positive rate:",
    f"{features['churn_90d'].mean():.4%}"
)

Feature store saved successfully.
Path: ..\data\processed\features.parquet
Shape: (424607, 101)
Customers: 11716
Snapshots: 93
Positive labels: 23029
Positive rate: 5.4236%


In [269]:
# ============================================================
# Verify persisted feature store
# ============================================================

saved_features = pd.read_parquet(
    output_path
)

print("Persisted feature store:")
print("Shape:", saved_features.shape)
print(
    "Customers:",
    saved_features["customer_id"].nunique()
)
print(
    "Snapshots:",
    saved_features["snapshot_date"].nunique()
)
print(
    "Positive labels:",
    saved_features["churn_90d"].sum()
)
print(
    "Positive rate:",
    f"{saved_features['churn_90d'].mean():.4%}"
)

assert saved_features.shape == features.shape
assert saved_features["customer_id"].nunique() == 11716
assert saved_features["snapshot_date"].nunique() == 93
assert saved_features["churn_90d"].sum() == 23029

print("\nPersisted feature store verification PASSED.")

Persisted feature store:
Shape: (424607, 101)
Customers: 11716
Snapshots: 93
Positive labels: 23029
Positive rate: 5.4236%

Persisted feature store verification PASSED.


In [271]:
import pandas as pd

features = pd.read_parquet(
    "../data/processed/features.parquet"
)

snapshot_dates = (
    pd.Series(
        pd.to_datetime(
            features["snapshot_date"]
        ).unique()
    )
    .sort_values()
    .reset_index(drop=True)
)

print("Number of snapshots:", len(snapshot_dates))

print("\nFirst snapshot:")
print(snapshot_dates.iloc[0])

print("\nLast snapshot:")
print(snapshot_dates.iloc[-1])

print("\n90 days before last snapshot:")
print(
    snapshot_dates.iloc[-1]
    - pd.Timedelta(days=90)
)

print("\n180 days before last snapshot:")
print(
    snapshot_dates.iloc[-1]
    - pd.Timedelta(days=180)
)

Number of snapshots: 93

First snapshot:
2024-01-01 00:00:00

Last snapshot:
2025-10-06 00:00:00

90 days before last snapshot:
2025-07-08 00:00:00

180 days before last snapshot:
2025-04-09 00:00:00


In [273]:
# ============================================================
# Temporal split design
# ============================================================

import pandas as pd


# Load frozen feature store
features = pd.read_parquet(
    "../data/processed/features.parquet"
)

features["snapshot_date"] = pd.to_datetime(
    features["snapshot_date"]
)


# ------------------------------------------------------------
# Unique weekly snapshots
# ------------------------------------------------------------

snapshot_dates = (
    pd.Series(
        features["snapshot_date"].unique()
    )
    .sort_values()
    .reset_index(drop=True)
)


print("Total snapshots:", len(snapshot_dates))


# ------------------------------------------------------------
# Proposed positions
# ------------------------------------------------------------

TRAIN_END_IDX = 43
PURGE_1_END_IDX = 56
VAL_END_IDX = 66
PURGE_2_END_IDX = 79
TEST_END_IDX = 92


split_dates = pd.DataFrame({
    "position": [
        0,
        TRAIN_END_IDX,
        TRAIN_END_IDX + 1,
        PURGE_1_END_IDX,
        PURGE_1_END_IDX + 1,
        VAL_END_IDX,
        VAL_END_IDX + 1,
        PURGE_2_END_IDX,
        PURGE_2_END_IDX + 1,
        TEST_END_IDX,
    ],
    "date": [
        snapshot_dates.iloc[0],
        snapshot_dates.iloc[TRAIN_END_IDX],
        snapshot_dates.iloc[TRAIN_END_IDX + 1],
        snapshot_dates.iloc[PURGE_1_END_IDX],
        snapshot_dates.iloc[PURGE_1_END_IDX + 1],
        snapshot_dates.iloc[VAL_END_IDX],
        snapshot_dates.iloc[VAL_END_IDX + 1],
        snapshot_dates.iloc[PURGE_2_END_IDX],
        snapshot_dates.iloc[PURGE_2_END_IDX + 1],
        snapshot_dates.iloc[TEST_END_IDX],
    ],
})


print("\nSplit boundaries:")
print(split_dates.to_string(index=False))

Total snapshots: 93

Split boundaries:
 position       date
        0 2024-01-01
       43 2024-10-28
       44 2024-11-04
       56 2025-01-27
       57 2025-02-03
       66 2025-04-07
       67 2025-04-14
       79 2025-07-07
       80 2025-07-14
       92 2025-10-06


In [274]:
# ============================================================
# Create temporal datasets
# ============================================================

train_end = snapshot_dates.iloc[TRAIN_END_IDX]

purge_1_start = snapshot_dates.iloc[TRAIN_END_IDX + 1]
purge_1_end = snapshot_dates.iloc[PURGE_1_END_IDX]

val_start = snapshot_dates.iloc[PURGE_1_END_IDX + 1]
val_end = snapshot_dates.iloc[VAL_END_IDX]

purge_2_start = snapshot_dates.iloc[VAL_END_IDX + 1]
purge_2_end = snapshot_dates.iloc[PURGE_2_END_IDX]

test_start = snapshot_dates.iloc[PURGE_2_END_IDX + 1]
test_end = snapshot_dates.iloc[TEST_END_IDX]


train = features[
    features["snapshot_date"] <= train_end
].copy()


validation = features[
    (
        features["snapshot_date"] >= val_start
    )
    &
    (
        features["snapshot_date"] <= val_end
    )
].copy()


test = features[
    (
        features["snapshot_date"] >= test_start
    )
    &
    (
        features["snapshot_date"] <= test_end
    )
].copy()


purge_1 = features[
    (
        features["snapshot_date"] >= purge_1_start
    )
    &
    (
        features["snapshot_date"] <= purge_1_end
    )
].copy()


purge_2 = features[
    (
        features["snapshot_date"] >= purge_2_start
    )
    &
    (
        features["snapshot_date"] <= purge_2_end
    )
].copy()


# ============================================================
# Report
# ============================================================

def report_split(name, df):

    print(f"\n{name}")
    print("-" * 50)

    print("Rows:", f"{len(df):,}")
    print(
        "Customers:",
        f"{df['customer_id'].nunique():,}"
    )
    print(
        "Snapshots:",
        df["snapshot_date"].nunique()
    )
    print(
        "Start:",
        df["snapshot_date"].min()
    )
    print(
        "End:",
        df["snapshot_date"].max()
    )
    print(
        "Positive labels:",
        f"{df['churn_90d'].sum():,}"
    )
    print(
        "Churn rate:",
        f"{df['churn_90d'].mean():.4%}"
    )


report_split("TRAIN", train)
report_split("PURGE 1", purge_1)
report_split("VALIDATION", validation)
report_split("PURGE 2", purge_2)
report_split("TEST", test)


TRAIN
--------------------------------------------------
Rows: 112,685
Customers: 5,441
Snapshots: 44
Start: 2024-01-01 00:00:00
End: 2024-10-28 00:00:00
Positive labels: 6,080
Churn rate: 5.3956%

PURGE 1
--------------------------------------------------
Rows: 73,570
Customers: 6,602
Snapshots: 13
Start: 2024-11-04 00:00:00
End: 2025-01-27 00:00:00
Positive labels: 3,897
Churn rate: 5.2970%

VALIDATION
--------------------------------------------------
Rows: 62,347
Customers: 7,243
Snapshots: 10
Start: 2025-02-03 00:00:00
End: 2025-04-07 00:00:00
Positive labels: 3,435
Churn rate: 5.5095%

PURGE 2
--------------------------------------------------
Rows: 85,647
Customers: 7,948
Snapshots: 13
Start: 2025-04-14 00:00:00
End: 2025-07-07 00:00:00
Positive labels: 4,654
Churn rate: 5.4339%

TEST
--------------------------------------------------
Rows: 90,358
Customers: 8,321
Snapshots: 13
Start: 2025-07-14 00:00:00
End: 2025-10-06 00:00:00
Positive labels: 4,963
Churn rate: 5.4926%


In [275]:
# ============================================================
# Temporal leakage checks
# ============================================================

print("\n" + "=" * 70)
print("TEMPORAL LEAKAGE CHECKS")
print("=" * 70)


# ------------------------------------------------------------
# Chronological ordering
# ------------------------------------------------------------

assert train["snapshot_date"].max() < validation["snapshot_date"].min()

assert validation["snapshot_date"].max() < test["snapshot_date"].min()


# ------------------------------------------------------------
# Purge boundaries
# ------------------------------------------------------------

assert train["snapshot_date"].max() < purge_1["snapshot_date"].min()
assert purge_1["snapshot_date"].max() < validation["snapshot_date"].min()

assert validation["snapshot_date"].max() < purge_2["snapshot_date"].min()
assert purge_2["snapshot_date"].max() < test["snapshot_date"].min()


# ------------------------------------------------------------
# No overlapping snapshot dates
# ------------------------------------------------------------

assert set(train["snapshot_date"]).isdisjoint(
    validation["snapshot_date"]
)

assert set(train["snapshot_date"]).isdisjoint(
    test["snapshot_date"]
)

assert set(validation["snapshot_date"]).isdisjoint(
    test["snapshot_date"]
)


# ------------------------------------------------------------
# Target horizon checks
# ------------------------------------------------------------

# Training labels must finish before validation starts.
train_label_end = (
    train["snapshot_date"].max()
    + pd.Timedelta(days=90)
)

print(
    "\nLatest TRAIN target horizon:",
    train_label_end
)

print(
    "Validation starts:",
    validation["snapshot_date"].min()
)

assert train_label_end < validation["snapshot_date"].min()


# Validation labels must finish before test starts.
validation_label_end = (
    validation["snapshot_date"].max()
    + pd.Timedelta(days=90)
)

print(
    "\nLatest VALIDATION target horizon:",
    validation_label_end
)

print(
    "Test starts:",
    test["snapshot_date"].min()
)

assert validation_label_end < test["snapshot_date"].min()


print(
    "\nTemporal leakage checks PASSED."
)


TEMPORAL LEAKAGE CHECKS

Latest TRAIN target horizon: 2025-01-26 00:00:00
Validation starts: 2025-02-03 00:00:00

Latest VALIDATION target horizon: 2025-07-06 00:00:00
Test starts: 2025-07-14 00:00:00

Temporal leakage checks PASSED.


In [279]:
from pathlib import Path

config_path = Path("../configs/split_config.json")

print(config_path.resolve())
print("Exists:", config_path.exists())

C:\Users\yadav\OneDrive\Desktop\Projects\Retain-AI\configs\split_config.json
Exists: True


In [281]:
import json
from pathlib import Path

# Your notebook is inside the notebooks/ folder
config_path = Path("../configs/split_config.json")

# Create configs folder if needed
config_path.parent.mkdir(parents=True, exist_ok=True)

# Frozen temporal split configuration
split_config = {
    "feature_store": "data/processed/features.parquet",
    "label_horizon_days": 90,

    "train": {
        "start": "2024-01-01",
        "end": "2024-10-28",
        "snapshots": 44
    },

    "purge_1": {
        "start": "2024-11-04",
        "end": "2025-01-27",
        "snapshots": 13
    },

    "validation": {
        "start": "2025-02-03",
        "end": "2025-04-07",
        "snapshots": 10
    },

    "purge_2": {
        "start": "2025-04-14",
        "end": "2025-07-07",
        "snapshots": 13
    },

    "test": {
        "start": "2025-07-14",
        "end": "2025-10-06",
        "snapshots": 13
    }
}

# Write JSON
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(split_config, f, indent=4)

print("Saved successfully!")
print(f"Location: {config_path.resolve()}")
print(f"File size: {config_path.stat().st_size} bytes")

Saved successfully!
Location: C:\Users\yadav\OneDrive\Desktop\Projects\Retain-AI\configs\split_config.json
File size: 654 bytes


In [282]:
from pathlib import Path

print("Notebook working directory:")
print(Path.cwd())

print("\nConfig file:")
print(Path("../configs/split_config.json").resolve())

print("\nExists:", Path("../configs/split_config.json").exists())

Notebook working directory:
c:\Users\yadav\OneDrive\Desktop\Projects\Retain-AI\notebooks

Config file:
C:\Users\yadav\OneDrive\Desktop\Projects\Retain-AI\configs\split_config.json

Exists: True
